# 01 · Mathematical Foundations

**Part I — Foundations** · Estimated time: ~120 min

Deep learning is applied mathematics, and the mathematics is not exotic: linear algebra to move
data through layers, calculus to compute how a loss changes when parameters change, probability to
say what a loss *means*, and a little numerical analysis to make it all work in finite-precision
arithmetic. This notebook teaches exactly the subset you need, and nothing you do not, with NumPy
code for every concept. Every formula that appears later in the course is derived or at least
motivated here, so that when Notebook 02 · Neural Networks from Scratch (NumPy) asks you to
implement backpropagation, or Notebook 04 · Training Deep Networks: Optimization, Initialization,
Normalization, Regularization introduces Adam, or Notebook 11 · Autoencoders and Variational
Autoencoders derives the evidence lower bound, the tools will already be in your hands.

The notebook is deliberately NumPy-centred. PyTorch appears only where NumPy has no equivalent
(the `bfloat16` format in Section 5); everything else is done with arrays so that you see the
mathematics without a framework in between.

## Learning objectives

After working through this notebook you will be able to:

- Manipulate vectors, matrices and higher-order tensors as NumPy arrays, reason about shapes, and read a matrix as a linear map.
- Use norms, eigenvalues and the singular value decomposition, and derive principal component analysis from the SVD.
- Explain the derivative as a linear approximation, compute gradients and Jacobians, and apply the chain rule in matrix form with the shapes used by backpropagation.
- Implement a finite-difference gradient checker and use it to validate hand-derived gradients.
- Explain why gradient descent decreases a loss, implement it, and diagnose the effects of the learning rate and of ill-conditioning.
- Derive mean-squared error and cross-entropy from maximum likelihood, and compute entropy, cross-entropy and KL divergence.
- Implement a numerically stable softmax and log-softmax using the log-sum-exp trick.
- Predict the behaviour of floating-point arithmetic (overflow, underflow, cancellation), apply NumPy broadcasting rules, and vectorise loops.

## Prerequisites

Notebook 00 · Course Overview and Setup (for the environment and conventions). Mathematically,
high-school algebra and a first encounter with derivatives are enough; everything else is built here.

## Contents

1. Linear algebra
2. Calculus
3. Optimization basics
4. Probability and information theory
5. Numerical computation

Exercises · Summary · Further reading

In [ ]:
import math, os, random, time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.datasets import load_digits, load_sample_image, load_iris
from sklearn.decomposition import PCA

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)          # the modern NumPy random generator, used throughout

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | device: {device}"
      + (f" ({torch.cuda.get_device_name(0)})" if device.type == "cuda" else ""))

# FAST_RUN = True shrinks every training budget so the whole notebook runs in a few
# minutes on a CPU. Set it to False for the full experiments (recommended on a GPU).
FAST_RUN = False
DATA_DIR = Path("../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True
np.set_printoptions(precision=4, suppress=True)
print(f"NumPy {np.__version__}")

## 1. Linear algebra

### 1.1 Scalars, vectors, matrices and tensors

A **scalar** is a single number, $a \in \mathbb{R}$. A **vector** $\mathbf{x} \in \mathbb{R}^n$ is an
ordered list of $n$ numbers; in this course a vector is always a *column*, and we write $x_i$ for its
$i$-th entry. A **matrix** $\mathbf{A} \in \mathbb{R}^{m \times n}$ is a rectangular table of numbers
with $m$ rows and $n$ columns, entry $A_{ij}$ in row $i$ and column $j$. A **tensor** generalises this
to any number of indices: a colour image is a tensor with three indices (channel, row, column), and a
mini-batch of colour images has four. The number of indices is called the **rank** or **order** of
the tensor (this use of "rank" is unrelated to the rank of a matrix, an unfortunate collision that we
will live with by always saying "matrix rank" for the latter).

In NumPy all of these are `ndarray` objects, distinguished only by their **shape**, a tuple giving the
size along each axis. A scalar has shape `()`, a vector `(n,)`, a matrix `(m, n)`, an image batch
`(N, C, H, W)`. The shape is the first thing to check whenever something goes wrong, so the cell
below prints it for one object of each kind, together with the number of axes (`ndim`), the total
number of entries (`size`) and the data type (`dtype`).

One subtlety deserves attention from the start. Mathematically a column vector and an
$n \times 1$ matrix are the same thing, but in NumPy a 1-D array of shape `(n,)` and a 2-D array of
shape `(n, 1)` behave differently in matrix products and broadcasting. The rule of thumb: keep
vectors 1-D unless you specifically need a row or a column, and reshape explicitly when you do.

In [ ]:
s = np.float64(3.5)                                   # scalar
v = np.array([1.0, 2.0, 3.0])                         # vector, shape (3,)
M = np.array([[1.0, 2.0, 3.0],
              [4.0, 5.0, 6.0]])                       # matrix, shape (2, 3)
T = rng.standard_normal((4, 3, 32, 32))               # rank-4 tensor: 4 RGB images of 32 x 32

for name, a in [("scalar s", s), ("vector v", v), ("matrix M", M), ("tensor T", T)]:
    a = np.asarray(a)
    print(f"{name:<9} ndim={a.ndim}  shape={str(a.shape):<16} size={a.size:<6} dtype={a.dtype}")

print("\nM[0, 2] (row 0, column 2) =", M[0, 2])
print("M[1]    (row 1)           =", M[1])
print("M[:, 1] (column 1)        =", M[:, 1], " <- returned as a 1-D array, shape", M[:, 1].shape)

col = v.reshape(3, 1)       # explicit column vector
row = v.reshape(1, 3)       # explicit row vector
print("\ncol.shape =", col.shape, "| row.shape =", row.shape)
print("row @ col (inner product) has shape", (row @ col).shape, "and value", (row @ col)[0, 0])
print("col @ row (outer product) has shape", (col @ row).shape)
print("v @ v (1-D arrays)        gives a plain scalar:", v @ v)

The last three lines show the difference between the 1-D and 2-D worlds: `row @ col` produces a
$1 \times 1$ matrix, `col @ row` produces a $3 \times 3$ matrix (the **outer product**), and
`v @ v` between two 1-D arrays produces a plain scalar (the **inner product**). All three are
correct; the point is that shapes carry meaning, and NumPy follows them literally.

### 1.2 The dot product and geometry

The **dot product** (inner product) of two vectors $\mathbf{a}, \mathbf{b} \in \mathbb{R}^n$ is

$$
\mathbf{a}^\top \mathbf{b} \;=\; \sum_{i=1}^{n} a_i b_i .
$$

It is the single most important operation in this course: every neuron computes a dot product of its
input with its weight vector, and every matrix multiplication is a table of dot products. Its power
comes from its geometric meaning. The **Euclidean length** (norm) of a vector is
$\lVert \mathbf{a} \rVert_2 = \sqrt{\mathbf{a}^\top \mathbf{a}}$, and for the angle $\theta$ between
two vectors one can show that

$$
\mathbf{a}^\top \mathbf{b} \;=\; \lVert \mathbf{a} \rVert_2 \, \lVert \mathbf{b} \rVert_2 \cos\theta .
$$

Hence the dot product is positive when the vectors point in similar directions, zero when they are
**orthogonal** (perpendicular), and negative when they point in opposite directions. Dividing by the
two norms gives the **cosine similarity**, which is how Notebook 10 · NLP with Pretrained
Transformers (Hugging Face) compares word embeddings; the raw dot product itself, without the
normalisation, is how attention scores are formed in Notebook 09 · Attention and Transformers.

The same identity gives the **projection** of $\mathbf{b}$ onto the direction of $\mathbf{a}$: the
component of $\mathbf{b}$ along $\mathbf{a}$ has length $\lVert \mathbf{b} \rVert \cos\theta =
\mathbf{a}^\top\mathbf{b} / \lVert \mathbf{a} \rVert$, so the projection vector is

$$
\mathrm{proj}_{\mathbf{a}}(\mathbf{b}) \;=\; \frac{\mathbf{a}^\top \mathbf{b}}{\mathbf{a}^\top \mathbf{a}} \, \mathbf{a},
$$

and the remainder $\mathbf{b} - \mathrm{proj}_{\mathbf{a}}(\mathbf{b})$ is orthogonal to $\mathbf{a}$.
A neuron with weights $\mathbf{w}$ therefore "measures how much of $\mathbf{w}$'s direction is present
in the input", and a whole layer projects the input onto many directions at once. The cell computes
the angle and the projection for two vectors in the plane and draws them.

In [ ]:
a = np.array([3.0, 1.0])
b = np.array([1.0, 2.0])

dot = a @ b
cos_theta = dot / (np.linalg.norm(a) * np.linalg.norm(b))
theta_deg = np.degrees(np.arccos(cos_theta))
proj = (a @ b) / (a @ a) * a
residual = b - proj
print(f"a.b = {dot:.3f} | cos(theta) = {cos_theta:.4f} | theta = {theta_deg:.2f} degrees")
print(f"projection of b onto a = {proj} | residual = {residual} | residual.a = {residual @ a:.2e} (orthogonal)")

fig, ax = plt.subplots(figsize=(5, 5))
for vec, color, label in [(a, "C0", "a"), (b, "C1", "b"), (proj, "C2", "proj_a(b)")]:
    ax.annotate("", xy=vec, xytext=(0, 0), arrowprops=dict(arrowstyle="->", color=color, lw=2.2))
    ax.text(vec[0] * 1.05, vec[1] * 1.05 + 0.05, label, color=color, fontsize=12)
ax.plot([b[0], proj[0]], [b[1], proj[1]], "k--", lw=1, label="residual (orthogonal to a)")
ax.set_xlim(-0.5, 3.5); ax.set_ylim(-0.5, 3.0); ax.set_aspect("equal")
ax.set_title("Dot product, angle and projection"); ax.legend(loc="upper left")
plt.tight_layout(); plt.show()

The residual's dot product with $\mathbf{a}$ is zero to machine precision, confirming orthogonality.
Keep this picture in mind: "projection onto a direction" and "dot product with a weight vector" are
the same operation, and most of a neural network consists of many such projections in parallel.

### 1.3 The matrix–vector product as a linear map

A matrix $\mathbf{A} \in \mathbb{R}^{m \times n}$ multiplied by a vector $\mathbf{x} \in \mathbb{R}^n$
gives a vector $\mathbf{y} = \mathbf{A}\mathbf{x} \in \mathbb{R}^m$ with entries
$y_i = \sum_j A_{ij} x_j$, i.e. the $i$-th entry of the output is the dot product of the $i$-th row
of $\mathbf{A}$ with $\mathbf{x}$. There is a second, equally useful reading: $\mathbf{A}\mathbf{x}$
is a *weighted sum of the columns of $\mathbf{A}$*, with the entries of $\mathbf{x}$ as weights,
$\mathbf{A}\mathbf{x} = \sum_j x_j \mathbf{A}_{:,j}$. In particular the columns of $\mathbf{A}$ are the
images of the standard basis vectors: $\mathbf{A}\mathbf{e}_j = \mathbf{A}_{:,j}$.

The map $\mathbf{x} \mapsto \mathbf{A}\mathbf{x}$ is **linear**: $\mathbf{A}(\alpha \mathbf{x} + \beta \mathbf{y})
= \alpha \mathbf{A}\mathbf{x} + \beta \mathbf{A}\mathbf{y}$. Conversely every linear map between
finite-dimensional spaces is a matrix. Geometrically, a linear map sends straight lines to straight
lines and keeps the origin fixed; it can stretch, rotate, shear and reflect, but it cannot bend. The
best way to build intuition is to watch what a $2 \times 2$ matrix does to a grid of points and to the
unit circle. The cell below draws six maps. Note the **determinant** in each title: it is the factor
by which areas are scaled (negative when orientation is flipped), and a determinant of zero means
the map squashes the plane onto a line, losing information irrecoverably.

In [ ]:
def draw_linear_map(A, ax, title, n_lines=7):
    """Draw the image of a square grid, the unit circle and the basis vectors under x -> A x."""
    ticks = np.linspace(-1, 1, n_lines)
    for t in ticks:
        horizontal = np.stack([ticks, np.full(n_lines, t)])       # (2, n): points on a horizontal line
        vertical = np.stack([np.full(n_lines, t), ticks])         # (2, n): points on a vertical line
        for line, color in [(horizontal, "C0"), (vertical, "C1")]:
            image = A @ line                                       # (2, 2) @ (2, n) -> (2, n)
            ax.plot(image[0], image[1], color=color, lw=0.8, alpha=0.7)
    angles = np.linspace(0, 2 * np.pi, 200)
    circle = A @ np.stack([np.cos(angles), np.sin(angles)])
    ax.plot(circle[0], circle[1], "k", lw=1.5)
    for basis, color in [(np.array([1.0, 0.0]), "C3"), (np.array([0.0, 1.0]), "C2")]:
        ax.annotate("", xy=A @ basis, xytext=(0, 0), arrowprops=dict(arrowstyle="->", color=color, lw=2))
    ax.set_aspect("equal"); ax.set_xlim(-2.6, 2.6); ax.set_ylim(-2.6, 2.6)
    ax.set_title(title, fontsize=9)

phi = np.radians(30)
linear_maps = {
    "identity": np.eye(2),
    "scaling diag(2, 0.5)": np.diag([2.0, 0.5]),
    "rotation by 30 degrees": np.array([[np.cos(phi), -np.sin(phi)], [np.sin(phi), np.cos(phi)]]),
    "shear [[1, 1], [0, 1]]": np.array([[1.0, 1.0], [0.0, 1.0]]),
    "reflection diag(1, -1)": np.diag([1.0, -1.0]),
    "rank-1 [[1, 2], [0.5, 1]]": np.array([[1.0, 2.0], [0.5, 1.0]]),
}
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for (name, A), ax in zip(linear_maps.items(), axes.ravel()):
    draw_linear_map(A, ax, f"{name}\ndet = {np.linalg.det(A):.2f}")
plt.tight_layout(); plt.show()

The red and green arrows are the images of the basis vectors $\mathbf{e}_1$ and $\mathbf{e}_2$, which
are exactly the two columns of each matrix. The unit circle becomes an ellipse in every case (a circle
for rotations and the identity, a line segment for the rank-1 matrix); the lengths and directions of
that ellipse's axes are what the singular value decomposition of Section 1.8 computes. The rank-1
matrix illustrates a **singular** map: its two columns are proportional, so the whole plane is
squashed onto a single line, the determinant is zero, and no inverse exists.

### 1.4 The matrix–matrix product and batching

Multiplying $\mathbf{A} \in \mathbb{R}^{m \times n}$ by $\mathbf{B} \in \mathbb{R}^{n \times p}$
gives $\mathbf{C} = \mathbf{A}\mathbf{B} \in \mathbb{R}^{m \times p}$ with
$C_{ik} = \sum_j A_{ij} B_{jk}$: entry $(i, k)$ is the dot product of row $i$ of $\mathbf{A}$ with
column $k$ of $\mathbf{B}$. The inner dimensions must agree ($n$ and $n$), and the outer dimensions
give the shape of the result; this "$(m \times n)(n \times p) = (m \times p)$" rule is the shape check
you will perform hundreds of times. Matrix multiplication is **associative**,
$(\mathbf{A}\mathbf{B})\mathbf{C} = \mathbf{A}(\mathbf{B}\mathbf{C})$, and distributive, but **not
commutative**: $\mathbf{A}\mathbf{B} \neq \mathbf{B}\mathbf{A}$ in general (the shapes may not even
allow both). It costs about $2mnp$ floating-point operations.

Why does deep learning care about matrix–matrix products when a neuron only needs a matrix–vector
product? Because of **batching**. If $\mathbf{W} \in \mathbb{R}^{d_{\text{in}} \times d_{\text{out}}}$
holds the weights of a layer, one input $\mathbf{x}$ gives $\mathbf{x}^\top \mathbf{W}$, and a batch
of $N$ inputs stacked as rows of $\mathbf{X} \in \mathbb{R}^{N \times d_{\text{in}}}$ gives all $N$
outputs at once: $\mathbf{H} = \mathbf{X}\mathbf{W} + \mathbf{b} \in \mathbb{R}^{N \times d_{\text{out}}}$,
where the bias row $\mathbf{b}$ is added to every row (Section 5.3 explains this **broadcasting**).
One large product instead of $N$ small ones is dramatically faster on modern hardware, as Section
5.4 measures. NumPy's `@` also handles *stacks* of matrices: two arrays of shape `(B, m, n)` and
`(B, n, p)` multiply pairwise to `(B, m, p)`, which is how attention heads are computed in
Notebook 09. The `np.einsum` function expresses the same product with an index formula that names
every axis, and is often the clearest way to write an unusual contraction.

The cell verifies these facts and ends with a small demonstration of associativity that matters in
practice: $\mathbf{A}(\mathbf{B}\mathbf{v})$ costs two matrix–vector products, about $4n^2$
operations, whereas $(\mathbf{A}\mathbf{B})\mathbf{v}$ first forms a full matrix product, about
$2n^3$ operations, roughly $n/2$ times more. The measured speed-up is smaller than the arithmetic
count predicts, because the matrix product is the most optimised routine in existence while
matrix–vector products are limited by memory bandwidth, but it is still one to two orders of
magnitude. (The cell also defines a small `best_of` timing helper, used again in Section 5.4, that
takes the minimum over several repeats so that one-off effects such as thread start-up do not
distort the numbers.)

In [ ]:
N, d_in, d_out = 5, 3, 4
X = rng.standard_normal((N, d_in))          # a batch of 5 inputs with 3 features
W = rng.standard_normal((d_in, d_out))      # weights of a layer 3 -> 4
b = rng.standard_normal(d_out)              # bias, one per output unit

H = X @ W + b                                # (5, 3) @ (3, 4) + (4,) -> (5, 4)
H_rows = np.stack([X[i] @ W + b for i in range(N)])
print("batched output shape:", H.shape, "| equals row-by-row computation:", np.allclose(H, H_rows))

A2 = rng.standard_normal((2, 2)); B2 = rng.standard_normal((2, 2))
print("A B == B A ?", np.allclose(A2 @ B2, B2 @ A2), "  (matrix multiplication is not commutative)")

stackA = rng.standard_normal((8, 3, 4)); stackB = rng.standard_normal((8, 4, 2))
C_stack = stackA @ stackB
C_einsum = np.einsum("bij,bjk->bik", stackA, stackB)
C_loop = np.stack([stackA[i] @ stackB[i] for i in range(8)])
print("stacked product shape:", C_stack.shape, "| einsum agrees:", np.allclose(C_stack, C_einsum),
      "| loop agrees:", np.allclose(C_stack, C_loop))

def best_of(fn, repeats=5):
    """Minimum wall-clock seconds of fn() over several repeats: the most reliable timing on a busy machine."""
    times = []
    for _ in range(repeats):
        t0 = time.perf_counter(); fn(); times.append(time.perf_counter() - t0)
    return min(times)

n = 1000
A_big, B_big, v_big = rng.standard_normal((n, n)), rng.standard_normal((n, n)), rng.standard_normal(n)
r1, r2 = (A_big @ B_big) @ v_big, A_big @ (B_big @ v_big)
t_left = best_of(lambda: (A_big @ B_big) @ v_big)
t_right = best_of(lambda: A_big @ (B_big @ v_big))
print(f"(A B) v: {t_left * 1e3:7.2f} ms | A (B v): {t_right * 1e3:7.3f} ms | same result: {np.allclose(r1, r2)}"
      f" | speed-up {t_left / t_right:.0f}x  (arithmetic count predicts about {(2 * n ** 3 + 2 * n ** 2) / (4 * n ** 2):.0f}x)")

Same numbers, wildly different cost: bracketing matters. The lesson generalises to the whole of
backpropagation, which is nothing but the chain rule bracketed in the cheap order (Section 2.4).

### 1.5 Transpose, identity and inverse

The **transpose** $\mathbf{A}^\top$ swaps rows and columns, $(\mathbf{A}^\top)_{ij} = A_{ji}$, and
reverses the order of a product: $(\mathbf{A}\mathbf{B})^\top = \mathbf{B}^\top \mathbf{A}^\top$. A
matrix with $\mathbf{A}^\top = \mathbf{A}$ is **symmetric**; covariance matrices and Hessians are
symmetric, and that structure is what makes their eigenvalues real (Section 1.7). The **identity**
$\mathbf{I}$ has ones on the diagonal and zeros elsewhere, so $\mathbf{I}\mathbf{x} = \mathbf{x}$. The
**inverse** $\mathbf{A}^{-1}$ of a square matrix, when it exists, satisfies $\mathbf{A}^{-1}\mathbf{A}
= \mathbf{A}\mathbf{A}^{-1} = \mathbf{I}$; it exists exactly when $\det \mathbf{A} \neq 0$, that is,
when the map does not squash any direction to zero. A matrix whose transpose is its inverse,
$\mathbf{Q}^\top \mathbf{Q} = \mathbf{I}$, is **orthogonal**: its columns are unit vectors that are
mutually perpendicular, and it represents a rotation or reflection that preserves lengths and
angles. Orthogonal matrices are the well-behaved citizens of numerical linear algebra and appear in
the SVD and in PCA.

Two practical remarks. First, in deep learning we almost never compute an inverse explicitly. To
solve $\mathbf{A}\mathbf{x} = \mathbf{b}$ one uses `np.linalg.solve(A, b)`, which is faster and more
accurate than forming $\mathbf{A}^{-1}$ and multiplying; and the parameters of a network are found by
gradient descent, not by solving equations. Second, near-singular matrices are as dangerous as
singular ones: the inverse exists but is enormous, and tiny rounding errors in the input are
amplified by the **condition number** $\kappa(\mathbf{A}) = \lVert \mathbf{A} \rVert \, \lVert
\mathbf{A}^{-1} \rVert$. We will meet the condition number again as the property of a loss surface
that decides how hard gradient descent has to work (Section 3.5).

In [ ]:
A = np.array([[2.0, 1.0],
              [1.0, 3.0]])
B = rng.standard_normal((2, 2))
print("(A B)^T == B^T A^T :", np.allclose((A @ B).T, B.T @ A.T))
print("A symmetric        :", np.allclose(A, A.T))

A_inv = np.linalg.inv(A)
print("\nA^-1 =\n", A_inv)
print("A @ A^-1 =\n", A @ A_inv, "\n(the identity, up to rounding)")

rhs = np.array([1.0, 2.0])
x_inv = A_inv @ rhs
x_solve = np.linalg.solve(A, rhs)
print("solution of A x = b via inverse:", x_inv, "| via solve:", x_solve, "| A x - b =", A @ x_solve - rhs)

Q = np.array([[np.cos(phi), -np.sin(phi)], [np.sin(phi), np.cos(phi)]])     # the rotation from 1.3
print("\nrotation is orthogonal: Q^T Q = I ->", np.allclose(Q.T @ Q, np.eye(2)),
      "| lengths preserved:", np.isclose(np.linalg.norm(Q @ rhs), np.linalg.norm(rhs)))

singular = linear_maps["rank-1 [[1, 2], [0.5, 1]]"]
try:
    np.linalg.inv(singular)
except np.linalg.LinAlgError as err:
    print("\ninverting the rank-1 matrix fails:", err)

eps = 1e-9
nearly_singular = np.array([[1.0, 1.0], [1.0, 1.0 + eps]])
print(f"nearly singular matrix: det = {np.linalg.det(nearly_singular):.1e}, "
      f"condition number = {np.linalg.cond(nearly_singular):.1e}")
print("its inverse has entries of size", np.abs(np.linalg.inv(nearly_singular)).max())

The nearly singular matrix has an inverse with entries around $10^9$: any error of $10^{-9}$ in the
right-hand side changes the solution by about $1$. That is what a condition number of a few billion
means, and it is why "just invert it" is bad advice.

### 1.6 Norms, and why they matter for regularisation

A **norm** assigns a length to a vector. The family of $\ell_p$ norms is

$$
\lVert \mathbf{x} \rVert_p = \Big( \sum_i |x_i|^p \Big)^{1/p}, \qquad
\lVert \mathbf{x} \rVert_1 = \sum_i |x_i|, \quad
\lVert \mathbf{x} \rVert_2 = \sqrt{\sum_i x_i^2}, \quad
\lVert \mathbf{x} \rVert_\infty = \max_i |x_i| .
$$

For matrices the most common norm in deep learning is the **Frobenius norm**, the $\ell_2$ norm of the
matrix flattened into a vector, $\lVert \mathbf{W} \rVert_F = \sqrt{\sum_{ij} W_{ij}^2} =
\sqrt{\mathrm{tr}(\mathbf{W}^\top \mathbf{W})}$, which also equals the square root of the sum of the
squared singular values (Section 1.8).

Norms matter because **regularisation** penalises the size of the parameters. Adding
$\lambda \lVert \mathbf{w} \rVert_2^2$ to the loss (weight decay, Notebook 04 · Training Deep
Networks: Optimization, Initialization, Normalization, Regularization) pulls all weights
smoothly towards zero; adding $\lambda \lVert \mathbf{w} \rVert_1$ instead drives many weights to
*exactly* zero, producing sparse models. The reason is geometric and visible in the picture below:
the $\ell_1$ unit ball is a diamond whose corners lie on the axes, so the point of the ball closest to
a typical loss minimum tends to be a corner, where some coordinates vanish; the $\ell_2$ ball is round
and has no preferred points. The $\ell_\infty$ ball is a square, and its norm is what bounds the size
of an adversarial perturbation in Notebook 16 · Interpretability, Robustness and Uncertainty. The
cell computes the norms of a vector and draws the
three unit balls as the level set $\lVert \mathbf{x} \rVert_p = 1$.

In [ ]:
x = np.array([3.0, -4.0, 0.5])
print(f"L1 = {np.linalg.norm(x, 1):.3f} | L2 = {np.linalg.norm(x, 2):.3f} | "
      f"Linf = {np.linalg.norm(x, np.inf):.3f}  (L1 >= L2 >= Linf always)")
Wm = rng.standard_normal((4, 3))
print(f"Frobenius norm of W: {np.linalg.norm(Wm, 'fro'):.4f} | sqrt(sum of squares): {np.sqrt((Wm ** 2).sum()):.4f} "
      f"| sqrt(sum of squared singular values): {np.sqrt((np.linalg.svd(Wm, compute_uv=False) ** 2).sum()):.4f}")

grid = np.linspace(-1.5, 1.5, 301)
gx, gy = np.meshgrid(grid, grid)
fig, ax = plt.subplots(figsize=(5, 5))
for p, color, label in [(1, "C0", r"$\ell_1$ (diamond)"), (2, "C1", r"$\ell_2$ (circle)"), (np.inf, "C2", r"$\ell_\infty$ (square)")]:
    norm_values = np.linalg.norm(np.stack([gx, gy], axis=-1), ord=p, axis=-1)
    ax.contour(gx, gy, norm_values, levels=[1.0], colors=color, linewidths=2)
    ax.plot([], [], color=color, lw=2, label=label)
ax.set_aspect("equal"); ax.set_xlabel("$x_1$"); ax.set_ylabel("$x_2$")
ax.set_title(r"Unit balls $\{x : \|x\|_p = 1\}$"); ax.legend(loc="upper right")
plt.tight_layout(); plt.show()

The printed inequality $\lVert \mathbf{x} \rVert_1 \ge \lVert \mathbf{x} \rVert_2 \ge \lVert \mathbf{x} \rVert_\infty$
is visible in the picture as the nesting of the balls: the diamond sits inside the circle, which sits
inside the square. The three ways of computing the Frobenius norm agree, and the third one, via the
singular values, will make sense after Section 1.8.

### 1.7 Eigenvalues and eigenvectors

For a square matrix $\mathbf{A}$, a non-zero vector $\mathbf{v}$ is an **eigenvector** with
**eigenvalue** $\lambda$ if

$$
\mathbf{A}\mathbf{v} = \lambda \mathbf{v},
$$

that is, the map only *stretches* $\mathbf{v}$ (by $\lambda$, possibly flipping it if $\lambda < 0$)
without rotating it. Eigenvectors are the directions along which a linear map acts like plain
multiplication, and once you know them the map is easy to understand: write any vector as a
combination of eigenvectors and each component simply gets scaled.

For a **symmetric** matrix the situation is as good as it gets: all eigenvalues are real, and the
eigenvectors can be chosen orthonormal, so that $\mathbf{A} = \mathbf{Q}\boldsymbol{\Lambda}\mathbf{Q}^\top$
with $\mathbf{Q}$ orthogonal and $\boldsymbol{\Lambda}$ diagonal (the **spectral theorem**). This
is the case that matters most in this course, because the two symmetric matrices we constantly meet
are the **covariance matrix** of a dataset (whose eigenvectors are the principal components, Section
1.9) and the **Hessian** of a loss (whose eigenvalues are the curvatures of the loss surface, Section
3.5). A symmetric matrix whose eigenvalues are all positive is **positive definite**; then
$\mathbf{x}^\top \mathbf{A} \mathbf{x} > 0$ for every non-zero $\mathbf{x}$, and the quadratic
$\frac12 \mathbf{x}^\top \mathbf{A} \mathbf{x}$ is a bowl whose axes are the eigenvectors and whose
steepness along each axis is the eigenvalue.

The cell computes the eigendecomposition of a symmetric $2 \times 2$ matrix with `np.linalg.eigh`
(the `h` stands for Hermitian, the complex generalisation of symmetric; use it whenever you know the
matrix is symmetric, as it is faster and guarantees real output), verifies the defining equation,
and draws the eigenvectors on top of the image of the unit circle: the ellipse's axes are exactly the
eigenvectors, stretched by the eigenvalues. It also runs **power iteration**, the simplest
eigenvalue algorithm: repeatedly multiplying a random vector by $\mathbf{A}$ and normalising
converges to the eigenvector with the largest $|\lambda|$, because that component grows fastest.
Finally it shows that a non-symmetric matrix (a rotation) can have complex eigenvalues, which is just
the algebraic way of saying that no real direction is left unrotated.

In [ ]:
A_sym = np.array([[2.0, 1.0],
                  [1.0, 2.0]])
eigenvalues, eigenvectors = np.linalg.eigh(A_sym)          # columns of `eigenvectors` are the eigenvectors
print("eigenvalues:", eigenvalues)
print("eigenvectors (as columns):\n", eigenvectors)
for lam, v in zip(eigenvalues, eigenvectors.T):
    print(f"  A v = {A_sym @ v}  and  lambda v = {lam * v}  -> equal: {np.allclose(A_sym @ v, lam * v)}")
Q, Lam = eigenvectors, np.diag(eigenvalues)
print("Q diag(lambda) Q^T reconstructs A:", np.allclose(Q @ Lam @ Q.T, A_sym), "| Q orthogonal:", np.allclose(Q.T @ Q, np.eye(2)))

# power iteration
v = rng.standard_normal(2)
for it in range(50):
    v = A_sym @ v
    v = v / np.linalg.norm(v)
print(f"power iteration -> direction {v}, Rayleigh quotient v^T A v = {v @ A_sym @ v:.6f} (largest eigenvalue)")

rotation_eigs = np.linalg.eigvals(linear_maps["rotation by 30 degrees"])
print("eigenvalues of the 30-degree rotation from 1.3:", rotation_eigs.round(4), " (complex: no real direction is fixed)")

fig, ax = plt.subplots(figsize=(5, 5))
draw_linear_map(A_sym, ax, "symmetric A: ellipse axes are the eigenvectors")
for lam, vec in zip(eigenvalues, eigenvectors.T):
    ax.annotate("", xy=lam * vec, xytext=(0, 0), arrowprops=dict(arrowstyle="->", color="magenta", lw=2.5))
    ax.text(*(lam * vec * 1.08), f"$\\lambda={lam:.0f}$", color="magenta", fontsize=11)
ax.set_xlim(-3.5, 3.5); ax.set_ylim(-3.5, 3.5)
plt.tight_layout(); plt.show()

The magenta arrows, the eigenvectors scaled by their eigenvalues, coincide with the axes of the
ellipse: the map stretches the diagonal direction by 3 and the anti-diagonal direction by 1. Power
iteration recovers the dominant direction to six decimals in fifty multiplications (the error shrinks
like $(\lambda_2 / \lambda_1)^t = 3^{-t}$). The same idea, applied to Hessian–vector products, is how
one estimates the largest curvature of a loss in practice, and it is how spectral normalisation
controls the Lipschitz constant of a GAN discriminator in Notebook 12 · GANs and Diffusion Models.

### 1.8 The singular value decomposition and low-rank approximation

Eigendecomposition needs a square matrix, and the nice version needs symmetry. The **singular value
decomposition** (SVD) works for *every* matrix $\mathbf{A} \in \mathbb{R}^{m \times n}$:

$$
\mathbf{A} = \mathbf{U} \boldsymbol{\Sigma} \mathbf{V}^\top,
\qquad \mathbf{U} \in \mathbb{R}^{m \times r},\;
\boldsymbol{\Sigma} = \mathrm{diag}(\sigma_1 \ge \sigma_2 \ge \dots \ge \sigma_r \ge 0),\;
\mathbf{V} \in \mathbb{R}^{n \times r},
$$

with $\mathbf{U}^\top \mathbf{U} = \mathbf{V}^\top \mathbf{V} = \mathbf{I}$ and $r = \min(m, n)$
(this is the "thin" form that `np.linalg.svd(..., full_matrices=False)` returns). The columns of
$\mathbf{V}$ are orthonormal input directions, the columns of $\mathbf{U}$ are orthonormal output
directions, and the **singular values** $\sigma_i$ say how much each input direction is stretched on
its way to the corresponding output direction. In the pictures of Section 1.3, the $\sigma_i$ are the
semi-axes of the ellipse. The number of non-zero singular values is the matrix rank. The SVD is
related to the eigendecomposition through $\mathbf{A}^\top \mathbf{A} = \mathbf{V}\boldsymbol{\Sigma}^2\mathbf{V}^\top$:
the right singular vectors are the eigenvectors of $\mathbf{A}^\top\mathbf{A}$ and
$\sigma_i^2$ are its eigenvalues, a fact we use to derive PCA in the next section.

Written as a sum, $\mathbf{A} = \sum_{i=1}^{r} \sigma_i \mathbf{u}_i \mathbf{v}_i^\top$ is a sum of
rank-one matrices with decreasing weights. Keeping only the first $k$ terms gives
$\mathbf{A}_k = \mathbf{U}_{:,:k} \boldsymbol{\Sigma}_{:k,:k} \mathbf{V}_{:,:k}^\top$, and the
**Eckart–Young theorem** states that this is the best rank-$k$ approximation of $\mathbf{A}$ in the
Frobenius norm, with error $\lVert \mathbf{A} - \mathbf{A}_k \rVert_F = \sqrt{\sum_{i > k} \sigma_i^2}$.
Storing $\mathbf{A}_k$ takes $k(m + n + 1)$ numbers instead of $mn$. This is the idea behind LoRA,
the parameter-efficient fine-tuning method of Notebook 10 · NLP with Pretrained Transformers
(Hugging Face), which adds a low-rank update to a large weight matrix, and behind the compression
of trained networks in Notebook 15 · Deployment and Model Optimization.

The first cell verifies the properties on a small random matrix; the second compresses a real
photograph (scikit-learn ships a sample image, so nothing is downloaded), converted to a grey-scale
matrix of shape $427 \times 640$.

In [ ]:
A_rect = rng.standard_normal((6, 4))
U, sigma, Vt = np.linalg.svd(A_rect, full_matrices=False)
print("shapes: U", U.shape, "| sigma", sigma.shape, "| V^T", Vt.shape)
print("singular values:", sigma)
print("U^T U = I:", np.allclose(U.T @ U, np.eye(4)), "| V^T V = I:", np.allclose(Vt @ Vt.T, np.eye(4)))
print("U diag(sigma) V^T reconstructs A:", np.allclose((U * sigma) @ Vt, A_rect))
eig_AtA = np.sort(np.linalg.eigvalsh(A_rect.T @ A_rect))[::-1]
print("eigenvalues of A^T A:", eig_AtA, "| sigma^2:", sigma ** 2)
print("Frobenius norm:", np.linalg.norm(A_rect, "fro").round(4), "| sqrt(sum sigma^2):", np.sqrt((sigma ** 2).sum()).round(4))

`(U * sigma) @ Vt` multiplies each column of $\mathbf{U}$ by the corresponding singular value
(broadcasting again), which is the same as `U @ np.diag(sigma) @ Vt` but avoids building the
diagonal matrix. Now the photograph.

In [ ]:
image = load_sample_image("china.jpg").mean(axis=2) / 255.0        # RGB (427, 640, 3) -> grey (427, 640)
m_px, n_px = image.shape
U_img, s_img, Vt_img = np.linalg.svd(image, full_matrices=False)

def low_rank_approx(k):
    """Best rank-k approximation of the image (Eckart-Young)."""
    return (U_img[:, :k] * s_img[:k]) @ Vt_img[:k]

ranks = [5, 20, 60]
fig, axes = plt.subplots(1, 4, figsize=(15, 3.6))
for k, ax in zip(ranks, axes[:3]):
    approx = low_rank_approx(k)
    rel_err = np.linalg.norm(image - approx, "fro") / np.linalg.norm(image, "fro")
    stored = k * (m_px + n_px + 1)
    ax.imshow(approx, cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"rank {k}: {stored / image.size:.1%} of the numbers,\nrelative error {rel_err:.3f}", fontsize=9)
    ax.axis("off")
axes[3].imshow(image, cmap="gray", vmin=0, vmax=1); axes[3].set_title(f"original (rank {len(s_img)})", fontsize=9); axes[3].axis("off")
plt.tight_layout(); plt.show()

ks = np.arange(1, len(s_img) + 1)
tail_energy = np.sqrt(np.cumsum(s_img[::-1] ** 2)[::-1])              # sqrt(sum_{i>=k} sigma_i^2)
rel_err_curve = np.append(tail_energy[1:], 0.0) / np.linalg.norm(image, "fro")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.6))
ax1.semilogy(ks, s_img); ax1.set_xlabel("index $i$"); ax1.set_ylabel(r"$\sigma_i$"); ax1.set_title("Singular values decay quickly")
ax2.plot(ks, rel_err_curve); ax2.set_xlabel("rank $k$"); ax2.set_ylabel("relative Frobenius error"); ax2.set_title("Error of the best rank-$k$ approximation")
plt.tight_layout(); plt.show()
k_check = 20
direct = np.linalg.norm(image - low_rank_approx(k_check), "fro") / np.linalg.norm(image, "fro")
print(f"rank {k_check}: error from the singular values {rel_err_curve[k_check - 1]:.6f} | measured directly {direct:.6f}")

With rank 20, storing under 8% of the numbers, the picture is clearly recognisable; with rank 60 it
is hard to distinguish from the original. The singular values fall off steeply, which is typical of
natural images and of many weight matrices in trained networks: most of the "energy" lives in a few
directions. The right-hand plot is the Eckart–Young formula in action, and the printed check confirms
that the error predicted from the discarded singular values equals the error measured directly.

### 1.9 Principal component analysis, derived from the SVD

**Principal component analysis** (PCA) finds the directions along which a dataset varies most, and
uses them to reduce the dimension of the data while losing as little as possible. It is the linear
ancestor of the representation learning that autoencoders perform in Notebook 11 · Autoencoders
and Variational Autoencoders (where we will show that a linear autoencoder recovers PCA), and it is the standard tool for looking at high-dimensional
data in two dimensions.

Let $\mathbf{X} \in \mathbb{R}^{N \times d}$ hold $N$ examples as rows, **centred** so that each column
has mean zero. The variance of the data along a unit direction $\mathbf{w}$ is the average squared
projection,

$$
\mathrm{Var}(\mathbf{X}\mathbf{w}) = \frac{1}{N-1} \lVert \mathbf{X}\mathbf{w} \rVert_2^2
= \mathbf{w}^\top \Big( \frac{\mathbf{X}^\top \mathbf{X}}{N-1} \Big) \mathbf{w}
= \mathbf{w}^\top \mathbf{C} \mathbf{w},
$$

where $\mathbf{C}$ is the $d \times d$ **covariance matrix**. The first principal direction maximises
$\mathbf{w}^\top \mathbf{C} \mathbf{w}$ subject to $\lVert \mathbf{w} \rVert = 1$. Since $\mathbf{C}$
is symmetric, write it as $\mathbf{C} = \mathbf{Q}\boldsymbol{\Lambda}\mathbf{Q}^\top$; substituting
$\mathbf{w} = \mathbf{Q}\mathbf{z}$ (with $\lVert \mathbf{z} \rVert = 1$ because $\mathbf{Q}$ is
orthogonal) turns the objective into $\sum_i \lambda_i z_i^2$, a weighted average of the eigenvalues,
which is maximised by putting all the weight on the largest one: $\mathbf{w} = \mathbf{q}_1$, the top
eigenvector, with variance $\lambda_1$. The second principal direction is the top eigenvector
orthogonal to the first, and so on.

Now the SVD does all of this at once. If $\mathbf{X} = \mathbf{U}\boldsymbol{\Sigma}\mathbf{V}^\top$, then
$\mathbf{C} = \mathbf{V}\boldsymbol{\Sigma}^2\mathbf{V}^\top / (N-1)$, so the principal directions are
the right singular vectors (columns of $\mathbf{V}$), the variance along the $i$-th one is
$\lambda_i = \sigma_i^2 / (N-1)$, and the coordinates of the data in the new basis, the
**scores**, are $\mathbf{Z} = \mathbf{X}\mathbf{V} = \mathbf{U}\boldsymbol{\Sigma}$. Keeping $k$
components gives the projection $\mathbf{Z}_k = \mathbf{X}\mathbf{V}_{:,:k}$ and the reconstruction
$\hat{\mathbf{X}} = \mathbf{Z}_k \mathbf{V}_{:,:k}^\top$ (plus the mean we subtracted), which by
Eckart–Young is the best rank-$k$ approximation of the centred data. The fraction of variance
explained by the first $k$ components is $\sum_{i \le k} \sigma_i^2 / \sum_i \sigma_i^2$. Computing
PCA through the SVD of $\mathbf{X}$ rather than the eigendecomposition of $\mathbf{C}$ is also
numerically preferable, because forming $\mathbf{X}^\top \mathbf{X}$ squares the condition number.

We demonstrate on scikit-learn's digits dataset: 1797 images of handwritten digits, each
$8 \times 8$ pixels flattened to a 64-dimensional vector. The cell centres the data, computes the
SVD, plots the first two scores coloured by digit, and shows the explained variance. As a check it
compares our result with scikit-learn's `PCA` (which may flip the sign of a component, since
$-\mathbf{v}$ is as good a direction as $\mathbf{v}$).

In [ ]:
X_digits, y_digits = load_digits(return_X_y=True)          # (1797, 64), labels 0..9
X_mean = X_digits.mean(axis=0)
X_centred = X_digits - X_mean                                # broadcasting: subtract the mean of each column
U_d, s_d, Vt_d = np.linalg.svd(X_centred, full_matrices=False)
components = Vt_d                                            # row i is the i-th principal direction (unit vector)
scores = X_centred @ components.T                            # (1797, 64), identical to U_d * s_d
explained = s_d ** 2 / (s_d ** 2).sum()
print("scores via X V equal U Sigma:", np.allclose(scores, U_d * s_d))
print("variance explained by the first 5 components:", explained[:5].round(3), "| first 2 total:", explained[:2].sum().round(3))

pca_sk = PCA(n_components=2).fit(X_centred)
sign_agnostic_match = np.allclose(np.abs(pca_sk.components_), np.abs(components[:2]), atol=1e-6)
print("matches sklearn PCA (up to sign):", sign_agnostic_match)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
sc = ax1.scatter(scores[:, 0], scores[:, 1], c=y_digits, cmap="tab10", s=8)
ax1.set_xlabel("PC 1"); ax1.set_ylabel("PC 2"); ax1.set_title("Digits projected on the first two principal components")
plt.colorbar(sc, ax=ax1, ticks=range(10), label="digit")
ax2.plot(np.arange(1, 65), np.cumsum(explained), marker=".")
ax2.axhline(0.9, color="gray", ls="--", lw=1); ax2.set_xlabel("number of components $k$"); ax2.set_ylabel("cumulative explained variance")
ax2.set_title("How many components are needed?")
plt.tight_layout(); plt.show()
print("components needed for 90% of the variance:", int(np.searchsorted(np.cumsum(explained), 0.9) + 1))

Two numbers out of sixty-four already separate several digit classes (the zeros, for instance, form
their own cluster), while visually similar digits overlap, as one would expect from their shapes.
About twenty components carry 90% of the variance, so the 64-pixel images effectively live near a
twenty-dimensional subspace. The final cell of this section makes the compression visible by
reconstructing a few digits from $k = 2, 5, 10, 20$ components and displaying the leading principal
directions themselves as $8 \times 8$ images.

In [ ]:
def pca_reconstruct(k, rows):
    """Reconstruct the given rows of the digits data from their first k principal components."""
    Z_k = X_centred[rows] @ components[:k].T          # (len(rows), k)
    return Z_k @ components[:k] + X_mean               # (len(rows), 64)

sample_rows = [0, 10, 20, 30, 40]
ks_show = [2, 5, 10, 20, 64]
fig, axes = plt.subplots(len(sample_rows), len(ks_show), figsize=(9, 8))
for r, row in enumerate(sample_rows):
    for c, k in enumerate(ks_show):
        axes[r, c].imshow(pca_reconstruct(k, [row]).reshape(8, 8), cmap="gray_r", vmin=0, vmax=16)
        axes[r, c].axis("off")
        if r == 0:
            axes[r, c].set_title(f"k = {k}", fontsize=10)
plt.suptitle("Digits reconstructed from k principal components (k = 64 is exact)"); plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 6, figsize=(11, 2.2))
for i, ax in enumerate(axes):
    ax.imshow(components[i].reshape(8, 8), cmap="RdBu"); ax.axis("off"); ax.set_title(f"PC {i + 1}", fontsize=10)
plt.suptitle("The leading principal directions are themselves 8 x 8 images"); plt.tight_layout(); plt.show()

for k in [2, 5, 10, 20]:
    err = np.linalg.norm(X_centred - (X_centred @ components[:k].T) @ components[:k], "fro") / np.linalg.norm(X_centred, "fro")
    unexplained = 1 - explained[:k].sum()
    print(f"k = {k:>2}: relative reconstruction error {err:.3f} | unexplained variance fraction {unexplained:.3f}, its square root {np.sqrt(unexplained):.3f}")

With two components the reconstructions are blurry averages; with ten they are recognisable; with
twenty they are close to the originals. The principal directions are interpretable as images: each
one is a pattern of positive and negative pixel weights (a stroke here, a gap there) that the data
vary along most, and the leading ones look like blurred contrasts between the centre and the ring
of the digit box. The relative reconstruction error is the square root of the unexplained variance
fraction, tying the
plot of the previous cell to the pictures. Exercise 1 repeats this analysis on a different dataset.

> **Key idea.** A matrix is a linear map; its SVD reveals the directions it stretches and by how
> much; keeping the largest stretches gives the best low-rank approximation, and applied to a centred
> data matrix that is exactly PCA.

## 2. Calculus

Training a network means adjusting millions of parameters to reduce a loss. Calculus answers the
only question that matters for that: *if I nudge this parameter a little, how much does the loss
change?* This section builds the answer from the one-variable derivative up to the matrix form of the
chain rule that backpropagation implements.

### 2.1 The derivative as the best linear approximation

The derivative of $f$ at $x_0$ is usually introduced as a limit,
$f'(x_0) = \lim_{h \to 0} \frac{f(x_0 + h) - f(x_0)}{h}$. The more useful way to think about it is as
an **approximation**: near $x_0$ the function is nearly a straight line,

$$
f(x_0 + h) \;=\; f(x_0) + f'(x_0)\, h + o(h),
$$

where $o(h)$ denotes a remainder that becomes negligible compared with $h$ as $h \to 0$ (for smooth
functions it is of order $h^2$). The derivative is the slope of the *best* linear approximation, and
"best" is what makes it unique. This is the viewpoint that generalises to many variables, where a
derivative becomes a linear map (a matrix) rather than a number, and it is the viewpoint that
explains gradient descent: if the loss is nearly linear in a small neighbourhood, then moving a
small step in the direction that decreases the linear approximation also decreases the loss.

The cell draws $f(x) = x^3 - 2x$ and its tangent at $x_0 = 1$ at three magnifications, and prints
the size of the remainder for a few step sizes.

In [ ]:
def f_cubic(x):
    return x ** 3 - 2 * x

def df_cubic(x):
    return 3 * x ** 2 - 2

x0 = 1.0
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, half_width in zip(axes, [2.0, 0.5, 0.05]):
    xs = np.linspace(x0 - half_width, x0 + half_width, 200)
    ax.plot(xs, f_cubic(xs), label="$f(x) = x^3 - 2x$")
    ax.plot(xs, f_cubic(x0) + df_cubic(x0) * (xs - x0), "--", label="tangent at $x_0 = 1$")
    ax.scatter([x0], [f_cubic(x0)], color="k", zorder=3)
    ax.set_title(f"window half-width {half_width}"); ax.set_xlabel("$x$")
axes[0].legend(); axes[0].set_ylabel("$f(x)$")
plt.suptitle("Zooming in, a smooth function becomes its tangent line"); plt.tight_layout(); plt.show()

for h in [0.1, 0.01, 0.001]:
    remainder = f_cubic(x0 + h) - (f_cubic(x0) + df_cubic(x0) * h)
    print(f"h = {h:<6} remainder = {remainder:.3e}   remainder / h = {remainder / h:.3e}   remainder / h^2 = {remainder / h**2:.3f}")

Dividing the step by ten divides the remainder by a hundred: the error of the linear approximation is
proportional to $h^2$ (the ratio in the last column tends to $f''(x_0) / 2 = 3$), so relative to the
step it vanishes. At the finest magnification the curve and the tangent are indistinguishable.

### 2.2 Numerical differentiation with central differences

We can turn the definition into an algorithm: pick a small $h$ and compute the difference quotient.
The **forward difference** $(f(x+h) - f(x)) / h$ has an error of order $h$, because the remainder we
just measured, of order $h^2$, is divided by $h$. A better idea is the **central difference**

$$
f'(x) \;\approx\; \frac{f(x + h) - f(x - h)}{2h} .
$$

Expanding both terms with Taylor's theorem, $f(x \pm h) = f(x) \pm f'(x) h + \tfrac12 f''(x) h^2 \pm
\tfrac16 f'''(x) h^3 + \dots$, the even-order terms cancel in the difference and the error is
$\tfrac16 f'''(x) h^2$, of order $h^2$ rather than $h$. So we should take $h$ tiny? Not quite. The
computer evaluates $f$ with a relative rounding error around $\varepsilon \approx 2 \times 10^{-16}$
in double precision, and dividing that error by $2h$ amplifies it to $\varepsilon / h$. The total
error $\approx h^2 + \varepsilon / h$ is minimised at $h \approx \varepsilon^{1/3} \approx 6 \times 10^{-6}$,
and below that the estimate gets *worse*. The cell measures both schemes on $f = \sin$ at $x = 1$
across fourteen orders of magnitude of $h$, and defines `numerical_gradient`, which applies the
central difference coordinate by coordinate to a function of an array. We will use it throughout
the rest of the notebook (and again in Notebook 02 · Neural Networks from Scratch (NumPy)) to
validate hand-derived gradients.

In [ ]:
def numerical_gradient(f, x, h=1e-5):
    """Central-difference gradient of a scalar-valued f at the array x (computed in float64)."""
    x = np.array(x, dtype=np.float64)               # private copy, so we can perturb entries in place
    grad = np.zeros_like(x)
    for idx in np.ndindex(x.shape):                  # iterate over every entry, whatever the shape
        original = x[idx]
        x[idx] = original + h; f_plus = f(x)
        x[idx] = original - h; f_minus = f(x)
        x[idx] = original
        grad[idx] = (f_plus - f_minus) / (2 * h)
    return grad

x_eval, true_derivative = 1.0, np.cos(1.0)
hs = np.logspace(-14, 0, 29)
forward_err = [abs((np.sin(x_eval + h) - np.sin(x_eval)) / h - true_derivative) for h in hs]
central_err = [abs((np.sin(x_eval + h) - np.sin(x_eval - h)) / (2 * h) - true_derivative) for h in hs]

plt.loglog(hs, forward_err, "o-", label="forward difference")
plt.loglog(hs, central_err, "s-", label="central difference")
plt.loglog(hs, hs / 6, "--", color="gray", lw=1, label=r"reference slopes $h$ and $h^2$")
plt.loglog(hs, hs ** 2 / 6, "--", color="gray", lw=1)
plt.loglog(hs, 2.2e-16 / hs, ":", color="k", lw=1, label=r"rounding error $\varepsilon / h$")
plt.xlabel("step size $h$"); plt.ylabel(r"absolute error of $f'(1)$, $f = \sin$")
plt.title("Truncation error versus rounding error"); plt.legend(fontsize=8); plt.tight_layout(); plt.show()

best = hs[int(np.argmin(central_err))]
idx_1e5 = int(np.argmin(np.abs(np.log10(hs) + 5)))                  # the grid point closest to h = 1e-5
print(f"best central-difference step: h = {best:.0e} (error {min(central_err):.1e}); default h = 1e-5 gives {central_err[idx_1e5]:.1e}")
print("numerical_gradient on a 2-D function f(x) = x1^2 + 3 x2^2 at (1, 2):", numerical_gradient(lambda p: p[0] ** 2 + 3 * p[1] ** 2, [1.0, 2.0]), "(exact: [2, 12])")

The V shape is the whole story of numerical differentiation. On the right the truncation error
dominates and follows the reference slopes: the central scheme (slope 2) beats the forward scheme
(slope 1) by many orders of magnitude at the same $h$. On the left both are drowned by rounding
error, which grows as $h$ shrinks. The sweet spot of the central scheme is around $h = 10^{-5}$ to
$10^{-6}$, with an error near $10^{-11}$, and that is the default step of `numerical_gradient`.
In `float32`, where $\varepsilon \approx 1.2 \times 10^{-7}$, the rounding-error line moves up by
nine orders of magnitude, the sweet spot moves to $h \approx \varepsilon^{1/3} \approx 5 \times 10^{-3}$,
and the best achievable error is around $10^{-4}$, far too coarse to distinguish a correct gradient
from a slightly wrong one; that is why gradient checks are always done in double precision.

> **Common pitfall.** Numerical differentiation is a *test*, not a *method*: to differentiate a
> network with a million parameters it needs two million forward passes. Backpropagation needs one
> backward pass. We use finite differences only to check that our analytic gradients are right.

### 2.3 Partial derivatives, the gradient, and steepest ascent

For a function of several variables, $f: \mathbb{R}^n \to \mathbb{R}$, the **partial derivative**
$\partial f / \partial x_i$ is the ordinary derivative with respect to $x_i$ while all other inputs are
held fixed. Collecting them into a vector gives the **gradient**

$$
\nabla f(\mathbf{x}) = \Big( \frac{\partial f}{\partial x_1}, \dots, \frac{\partial f}{\partial x_n} \Big)^\top,
$$

and the best linear approximation becomes $f(\mathbf{x} + \mathbf{h}) \approx f(\mathbf{x}) +
\nabla f(\mathbf{x})^\top \mathbf{h}$. The rate of change of $f$ along a unit direction $\mathbf{u}$,
the **directional derivative**, is therefore $D_{\mathbf{u}} f = \nabla f^\top \mathbf{u} =
\lVert \nabla f \rVert \cos\theta$, where $\theta$ is the angle between $\mathbf{u}$ and the gradient
(Section 1.2). This is maximal when $\theta = 0$: **the gradient points in the direction of steepest
ascent**, its negative points in the direction of steepest descent, and its length is the slope in
that direction. Directions perpendicular to the gradient have zero directional derivative, so the
gradient is perpendicular to the level sets (contours) of $f$.

The cell draws the contours of $f(x_1, x_2) = x_1^2 + 3x_2^2 + x_1 x_2$ with the gradient field on
top, and then verifies the cosine law for the directional derivative at one point by comparing
$\nabla f^\top \mathbf{u}$ with a numerical directional derivative for directions all around the
circle.

In [ ]:
def f_bowl(p):
    return p[0] ** 2 + 3 * p[1] ** 2 + p[0] * p[1]

def grad_bowl(p):
    return np.array([2 * p[0] + p[1], 6 * p[1] + p[0]])

g1, g2 = np.meshgrid(np.linspace(-3, 3, 61), np.linspace(-3, 3, 61))
Z = f_bowl(np.stack([g1, g2]))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.6))
ax1.contour(g1, g2, Z, levels=20, cmap="viridis")
sub = slice(None, None, 6)
G = grad_bowl(np.stack([g1[sub, sub], g2[sub, sub]]))
ax1.quiver(g1[sub, sub], g2[sub, sub], G[0], G[1], color="C3", alpha=0.8, width=0.004)
ax1.set_aspect("equal"); ax1.set_xlabel("$x_1$"); ax1.set_ylabel("$x_2$")
ax1.set_title("Contours of $f$ and its gradient field (perpendicular to contours)")

p0 = np.array([1.5, -1.0])
g0 = grad_bowl(p0)
angles = np.linspace(0, 2 * np.pi, 145)
directions = np.stack([np.cos(angles), np.sin(angles)], axis=1)                 # (145, 2) unit vectors
numeric_dd = np.array([(f_bowl(p0 + 1e-5 * u) - f_bowl(p0 - 1e-5 * u)) / 2e-5 for u in directions])
formula_dd = directions @ g0
ax2.plot(np.degrees(angles), numeric_dd, lw=3, alpha=0.5, label="numerical directional derivative")
ax2.plot(np.degrees(angles), formula_dd, "--", color="k", label=r"$\nabla f^\top u = \|\nabla f\| \cos\theta$")
gradient_angle = np.degrees(np.arctan2(g0[1], g0[0])) % 360
ax2.axvline(gradient_angle, color="C3", ls=":", label=f"direction of the gradient ({gradient_angle:.0f} degrees)")
ax2.set_xlabel("direction angle (degrees)"); ax2.set_ylabel("rate of change of $f$")
ax2.set_title(f"Directional derivatives at {tuple(p0.tolist())}"); ax2.legend(fontsize=8)
plt.tight_layout(); plt.show()
print(f"gradient at p0: {g0}, norm {np.linalg.norm(g0):.3f} = maximum of the curve {numeric_dd.max():.3f}")

The arrows cross the contour lines at right angles and get longer where the contours are closer
together, which is where the function is steeper. The right-hand panel is the cosine law: the rate
of change is a cosine wave in the direction angle, peaking at the gradient's direction with height
$\lVert \nabla f \rVert$, and crossing zero exactly where the direction is tangent to a contour.

### 2.4 The Jacobian and the chain rule in matrix form

Now let the function have several outputs, $\mathbf{f}: \mathbb{R}^n \to \mathbb{R}^m$, as every layer
of a network does. Its derivative is the **Jacobian matrix**

$$
\mathbf{J}_{\mathbf{f}}(\mathbf{x}) \in \mathbb{R}^{m \times n}, \qquad
(\mathbf{J}_{\mathbf{f}})_{ij} = \frac{\partial f_i}{\partial x_j},
$$

one row per output, one column per input, and the linear approximation reads $\mathbf{f}(\mathbf{x} +
\mathbf{h}) \approx \mathbf{f}(\mathbf{x}) + \mathbf{J}_{\mathbf{f}}(\mathbf{x})\, \mathbf{h}$. For a scalar
function the Jacobian is a $1 \times n$ row, the transpose of the gradient. For a linear layer
$\mathbf{h} = \mathbf{W}\mathbf{x} + \mathbf{b}$ the Jacobian with respect to $\mathbf{x}$ is simply
$\mathbf{W}$, and for an element-wise activation $\mathbf{a} = \phi(\mathbf{h})$ it is the diagonal
matrix $\mathrm{diag}(\phi'(\mathbf{h}))$.

The **chain rule** for compositions is matrix multiplication of Jacobians. If $\mathbf{y} =
\mathbf{g}(\mathbf{x})$ and $\mathbf{z} = \mathbf{h}(\mathbf{y})$, then

$$
\mathbf{J}_{\mathbf{h} \circ \mathbf{g}}(\mathbf{x}) \;=\; \mathbf{J}_{\mathbf{h}}(\mathbf{y}) \, \mathbf{J}_{\mathbf{g}}(\mathbf{x}),
\qquad (p \times n) = (p \times m)(m \times n).
$$

Now consider what a network does: a chain of layers ending in a scalar loss,
$L = \ell(\mathbf{h}_K)$, $\mathbf{h}_k = \mathbf{f}_k(\mathbf{h}_{k-1})$, $\mathbf{h}_0 = \mathbf{x}$.
The gradient of $L$ with respect to the input, written as a column vector, is the transpose of the
$1 \times n$ Jacobian of the whole chain,

$$
\nabla_{\mathbf{x}} L \;=\; \mathbf{J}_1^\top \, \mathbf{J}_2^\top \cdots \mathbf{J}_K^\top \, \nabla_{\mathbf{h}_K} \ell .
$$

Read from right to left this is a sequence of **vector–Jacobian products**: start with the gradient
of the loss with respect to the last activation (a vector), multiply by $\mathbf{J}_K^\top$ to get the
gradient with respect to the previous activation (another vector), and so on back to the input. Each
step is a matrix–vector product, and the full Jacobians are never multiplied together, exactly the
$\mathbf{A}(\mathbf{B}\mathbf{v})$ lesson of Section 1.4. That right-to-left evaluation *is*
backpropagation, and the reason it is cheap is that the loss is a scalar: the quantity flowing
backwards is always a vector, never a matrix. Notebook 03 · Tensors and Autograd builds an engine
that automates precisely this.

For a batched linear layer the shapes are the ones we will use in Notebook 02. Let
$\mathbf{H} = \mathbf{X}\mathbf{W} + \mathbf{b}$ with $\mathbf{X} \in \mathbb{R}^{N \times d_{\text{in}}}$,
$\mathbf{W} \in \mathbb{R}^{d_{\text{in}} \times d_{\text{out}}}$, and suppose the gradient of the loss with
respect to the output, $\mathbf{G} = \partial L / \partial \mathbf{H} \in \mathbb{R}^{N \times d_{\text{out}}}$,
is already known (it is what flows back from the layers above). Since $H_{nj} = \sum_i X_{ni} W_{ij} + b_j$,
the chain rule gives

$$
\frac{\partial L}{\partial W_{ij}} = \sum_n \frac{\partial L}{\partial H_{nj}} \frac{\partial H_{nj}}{\partial W_{ij}} = \sum_n G_{nj} X_{ni}
\;\;\Longrightarrow\;\;
\frac{\partial L}{\partial \mathbf{W}} = \mathbf{X}^\top \mathbf{G}, \qquad
\frac{\partial L}{\partial \mathbf{b}} = \sum_{n} \mathbf{G}_{n,:}, \qquad
\frac{\partial L}{\partial \mathbf{X}} = \mathbf{G}\mathbf{W}^\top .
$$

Each gradient has the shape of the thing it is taken with respect to, as it must:
$(d_{\text{in}} \times N)(N \times d_{\text{out}}) = d_{\text{in}} \times d_{\text{out}}$ for $\mathbf{W}$, and
$(N \times d_{\text{out}})(d_{\text{out}} \times d_{\text{in}}) = N \times d_{\text{in}}$ for $\mathbf{X}$. An
element-wise activation $\mathbf{A} = \phi(\mathbf{H})$ contributes the rule
$\partial L / \partial \mathbf{H} = \partial L / \partial \mathbf{A} \odot \phi'(\mathbf{H})$ (the diagonal
Jacobian becomes an element-wise product). The cell verifies all of these formulas numerically on
a one-layer network $L = \tfrac12 \lVert \tanh(\mathbf{X}\mathbf{W} + \mathbf{b}) - \mathbf{Y} \rVert_F^2$,
whose analytic gradients follow from the rules above with $\partial L/\partial \mathbf{A} =
\mathbf{A} - \mathbf{Y}$ and $\tanh'(h) = 1 - \tanh^2(h)$.

In [ ]:
N, d_in, d_out = 4, 3, 2
X0 = rng.standard_normal((N, d_in)); W0 = rng.standard_normal((d_in, d_out))
b0 = rng.standard_normal(d_out);     Y0 = rng.standard_normal((N, d_out))

def layer_forward(X, W, b):
    return np.tanh(X @ W + b)

def layer_loss(X, W, b):
    return 0.5 * np.sum((layer_forward(X, W, b) - Y0) ** 2)

def layer_grads(X, W, b):
    """Analytic gradients (dL/dX, dL/dW, dL/db) from the matrix chain rule."""
    A = layer_forward(X, W, b)          # (N, d_out)
    G_A = A - Y0                        # dL/dA, shape (N, d_out)
    G_H = G_A * (1 - A ** 2)            # dL/dH = dL/dA * tanh'(H), element-wise
    return G_H @ W.T, X.T @ G_H, G_H.sum(axis=0)

dX, dW, db = layer_grads(X0, W0, b0)
dX_num = numerical_gradient(lambda X: layer_loss(X, W0, b0), X0)
dW_num = numerical_gradient(lambda W: layer_loss(X0, W, b0), W0)
db_num = numerical_gradient(lambda b: layer_loss(X0, W0, b), b0)
for name, analytic, numeric, target in [("dL/dX", dX, dX_num, X0), ("dL/dW", dW, dW_num, W0), ("dL/db", db, db_num, b0)]:
    print(f"{name}: shape {str(analytic.shape):<8} = shape of the variable {str(target.shape):<8} | "
          f"max |analytic - numeric| = {np.abs(analytic - numeric).max():.2e}")

All three analytic gradients agree with the finite differences to about $10^{-10}$, and each has the
shape of its variable. These three lines, `G_H @ W.T`, `X.T @ G_H` and `G_H.sum(axis=0)`, are the
backward pass of a linear layer; Notebook 02 will wrap them in a class and stack them.

### 2.5 Gradients of three expressions you will meet constantly

**The linear function.** $f(\mathbf{w}) = \mathbf{x}^\top \mathbf{w} = \sum_i x_i w_i$ has
$\partial f / \partial w_i = x_i$, so $\nabla_{\mathbf{w}} f = \mathbf{x}$: the gradient of a dot
product with respect to one factor is the other factor. This is why the gradient of a neuron's
pre-activation with respect to its weights is its input.

**The squared error of a linear model.** $f(\mathbf{w}) = \lVert \mathbf{X}\mathbf{w} - \mathbf{y} \rVert_2^2$.
Write $\mathbf{r} = \mathbf{X}\mathbf{w} - \mathbf{y}$ (the residual), so $f = \mathbf{r}^\top \mathbf{r}$.
The gradient of $\mathbf{r}^\top\mathbf{r}$ with respect to $\mathbf{r}$ is $2\mathbf{r}$, and the
Jacobian of $\mathbf{r}$ with respect to $\mathbf{w}$ is $\mathbf{X}$; the chain rule (Jacobian
transposed times upstream gradient) gives

$$
\nabla_{\mathbf{w}} f = 2 \mathbf{X}^\top (\mathbf{X}\mathbf{w} - \mathbf{y}) .
$$

Setting it to zero yields the **normal equations** $\mathbf{X}^\top \mathbf{X} \mathbf{w} =
\mathbf{X}^\top \mathbf{y}$, whose solution is the least-squares fit; linear regression is the one
model in this course with a closed-form solution, and it makes a good test case for gradient descent
in Section 3.

**The softmax function.** $\mathbf{p} = \mathrm{softmax}(\mathbf{z})$ with $p_i = e^{z_i} / \sum_k e^{z_k}$
maps arbitrary scores (logits) to a probability vector, and is the output layer of every classifier
in the course. Writing $S = \sum_k e^{z_k}$ and using the quotient rule,

$$
\frac{\partial p_i}{\partial z_j} = \frac{\delta_{ij} e^{z_i} S - e^{z_i} e^{z_j}}{S^2} = p_i (\delta_{ij} - p_j),
\qquad \text{i.e.} \qquad
\mathbf{J}_{\mathrm{softmax}} = \mathrm{diag}(\mathbf{p}) - \mathbf{p}\mathbf{p}^\top ,
$$

where $\delta_{ij}$ is 1 if $i = j$ and 0 otherwise. The rows of this Jacobian sum to zero (the
probabilities always sum to one, so no change in $\mathbf{z}$ can change their total), and it is
symmetric. In Notebook 02 we combine it with the cross-entropy loss and find that the two together
have the astonishingly simple gradient $\mathbf{p} - \mathbf{y}$. The cell checks all three results
numerically; the softmax implementation subtracts the maximum logit first, for reasons explained in
Section 4.6.

In [ ]:
def softmax(z):
    """Numerically stable softmax of a 1-D array (see Section 4.6 for why we subtract the max)."""
    shifted = np.exp(z - z.max())
    return shifted / shifted.sum()

# (a) x^T w
x_vec, w_vec = rng.standard_normal(5), rng.standard_normal(5)
print("(a) grad of x^T w equals x:", np.allclose(numerical_gradient(lambda w: x_vec @ w, w_vec), x_vec))

# (b) ||X w - y||^2 and the normal equations
X_ls, y_ls = rng.standard_normal((20, 3)), rng.standard_normal(20)
sq_error = lambda w: np.sum((X_ls @ w - y_ls) ** 2)
analytic = 2 * X_ls.T @ (X_ls @ w_vec[:3] - y_ls)
print("(b) grad of ||Xw - y||^2 equals 2 X^T (Xw - y):", np.allclose(numerical_gradient(sq_error, w_vec[:3]), analytic))
w_normal = np.linalg.solve(X_ls.T @ X_ls, X_ls.T @ y_ls)
w_lstsq = np.linalg.lstsq(X_ls, y_ls, rcond=None)[0]
print("    normal-equation solution:", w_normal, "| np.linalg.lstsq:", w_lstsq,
      "| gradient at the solution:", np.abs(2 * X_ls.T @ (X_ls @ w_normal - y_ls)).max().round(12))

# (c) the softmax Jacobian
z = np.array([1.0, 2.0, 0.5, -1.0])
p = softmax(z)
J_analytic = np.diag(p) - np.outer(p, p)
J_numeric = np.stack([numerical_gradient(lambda zz, i=i: softmax(zz)[i], z) for i in range(len(z))])
print("(c) softmax Jacobian matches finite differences:", np.allclose(J_analytic, J_numeric, atol=1e-8),
      "| rows sum to zero:", np.allclose(J_analytic.sum(axis=1), 0), "| symmetric:", np.allclose(J_analytic, J_analytic.T))
print("    J =\n", J_analytic)

All three derivations check out. Look at how the softmax Jacobian scales with the probabilities:
every entry is a product of two probabilities, $p_i(1 - p_i)$ on the diagonal and $-p_i p_j$ off it,
so the entries involving the least likely class (the last logit, $p_4 \approx 0.03$) are already
small, and when one class holds nearly all the probability *every* entry is close to zero: no logit
can move the output much. That is the "saturation" of softmax, and it is one reason why a confident
wrong prediction learns slowly under a squared-error loss on the probabilities but quickly under
cross-entropy, whose logarithm cancels the saturation (Section 4.4 and Notebook 02).

### 2.6 A gradient checker

We now package the comparison into a function that we will use in Notebook 02 to test every layer of
a hand-written network. The **relative error**

$$
\frac{\lVert \nabla_{\text{analytic}} - \nabla_{\text{numeric}} \rVert_2}{\lVert \nabla_{\text{analytic}} \rVert_2 + \lVert \nabla_{\text{numeric}} \rVert_2}
$$

is scale-free, which matters because gradients range from $10^{-8}$ to $10^{8}$ and an absolute
tolerance would be meaningless. We read it with thresholds that are a slightly stricter version of
the ones suggested in the CS231n course notes: below $10^{-7}$ the gradient is right; between
$10^{-7}$ and $10^{-3}$ look carefully (kinks in ReLU-like functions, or a loss that is evaluated
in `float32` somewhere, can cause this legitimately); above $10^{-3}$ there is almost certainly a
bug. Always check in
`float64`, at random points (so that a coincidental zero does not hide an error), and with inputs of
a typical scale. The cell demonstrates the checker on the least-squares gradient with a correct
implementation, a version with a missing factor 2 (the most common mistake) and a version written as `r @ X`
instead of `X.T @ r`, which looks like a wrong transpose but is not.

In [ ]:
def gradient_check(f, grad_f, x, h=1e-5, name="gradient"):
    """Relative error between the analytic gradient grad_f(x) and central differences of f at x."""
    x = np.array(x, dtype=np.float64)
    analytic = np.asarray(grad_f(x), dtype=np.float64)
    numeric = numerical_gradient(f, x, h)
    if analytic.shape != numeric.shape:
        print(f"{name:<32} SHAPE MISMATCH: analytic {analytic.shape} vs numeric {numeric.shape}")
        return np.inf
    rel_error = np.linalg.norm(analytic - numeric) / (np.linalg.norm(analytic) + np.linalg.norm(numeric) + 1e-300)
    verdict = "correct" if rel_error < 1e-7 else ("suspicious" if rel_error < 1e-3 else "WRONG")
    print(f"{name:<32} relative error {rel_error:.2e} -> {verdict}")
    return rel_error

w_test = rng.standard_normal(3)
gradient_check(sq_error, lambda w: 2 * X_ls.T @ (X_ls @ w - y_ls), w_test, name="least squares, correct")
gradient_check(sq_error, lambda w: X_ls.T @ (X_ls @ w - y_ls), w_test, name="least squares, missing factor 2")
gradient_check(sq_error, lambda w: 2 * (X_ls @ w - y_ls) @ X_ls, w_test, name="least squares, (r @ X) form")   # same as X^T r: fine
gradient_check(lambda W: layer_loss(X0, W, b0), lambda W: layer_grads(X0, W, b0)[1], W0, name="tanh layer dL/dW")
gradient_check(lambda W: layer_loss(X0, W, b0), lambda W: 0.99 * layer_grads(X0, W, b0)[1], W0, name="tanh layer dL/dW, 1% too small")

The correct implementations score around $10^{-10}$, the missing factor scores $0.33$ (a third of
the magnitude is missing, and the relative error says so), and a gradient that is merely one percent
too small scores $5 \times 10^{-3}$: even small errors are unmistakable. Note that `r @ X` and
`X.T @ r` are the same computation with 1-D arrays, so the third check passes; the checker cares
about values, not about how you wrote them.

> **Key idea.** The derivative is a linear map; the chain rule multiplies those maps; for a scalar
> loss, evaluating the chain right to left as vector–Jacobian products is backpropagation. Check
> every hand-derived gradient with central differences in double precision.

## 3. Optimization basics

### 3.1 Taylor expansion and why moving against the gradient decreases the loss

Extend the linear approximation of Section 2 by one more term. For a twice-differentiable loss
$J(\boldsymbol{\theta})$ and a small displacement $\boldsymbol{\delta}$,

$$
J(\boldsymbol{\theta} + \boldsymbol{\delta}) \;=\; J(\boldsymbol{\theta}) + \nabla J^\top \boldsymbol{\delta}
+ \tfrac12 \boldsymbol{\delta}^\top \mathbf{H} \boldsymbol{\delta} + O(\lVert \boldsymbol{\delta} \rVert^3),
$$

where $\mathbf{H}$ is the **Hessian**, the symmetric matrix of second derivatives
$H_{ij} = \partial^2 J / \partial \theta_i \partial \theta_j$, which describes the curvature of the
loss surface. Now take the gradient descent step $\boldsymbol{\delta} = -\eta \nabla J =: -\eta \mathbf{g}$:

$$
J(\boldsymbol{\theta} - \eta \mathbf{g}) \;\approx\; J(\boldsymbol{\theta}) - \eta \lVert \mathbf{g} \rVert^2 + \tfrac12 \eta^2 \, \mathbf{g}^\top \mathbf{H} \mathbf{g} .
$$

The first-order term is $-\eta \lVert \mathbf{g} \rVert^2$, strictly negative whenever the gradient
is non-zero, so for a small enough step the loss *must* decrease; that is the entire justification of
gradient descent. The second-order term is a penalty that grows with $\eta^2$ and with the curvature
along the gradient. Balancing the two, the step that minimises the quadratic model is
$\eta^* = \lVert \mathbf{g} \rVert^2 / (\mathbf{g}^\top \mathbf{H} \mathbf{g})$, and the loss still decreases
for any $\eta < 2\eta^*$. Since $\mathbf{g}^\top \mathbf{H} \mathbf{g} \le \lambda_{\max} \lVert \mathbf{g} \rVert^2$,
a learning rate below $1 / \lambda_{\max}$ is always safe and one above $2 / \lambda_{\max}$ can make
things worse: the largest curvature of the loss caps the learning rate. We will see this bound
enforced to the decimal in the next section.

The cell illustrates the expansion on the **Rosenbrock function**
$f(x, y) = (1 - x)^2 + 100 (y - x^2)^2$, a classic test case with a curved, narrow valley, whose
gradient and Hessian we write down by hand (they are reused in Section 3.3). Starting from a point,
it plots the true loss along the negative gradient direction as a function of $\eta$, together with
the first-order (linear) and second-order (quadratic) predictions and the predicted best step.

In [ ]:
def rosenbrock(p):
    x, y = p
    return (1 - x) ** 2 + 100 * (y - x ** 2) ** 2

def rosenbrock_grad(p):
    x, y = p
    return np.array([-2 * (1 - x) - 400 * x * (y - x ** 2), 200 * (y - x ** 2)])

def rosenbrock_hess(p):
    x, y = p
    return np.array([[2 - 400 * y + 1200 * x ** 2, -400 * x], [-400 * x, 200.0]])

gradient_check(rosenbrock, rosenbrock_grad, [-1.2, 1.0], name="Rosenbrock gradient")
theta0 = np.array([-1.0, 1.5])
g, Hm = rosenbrock_grad(theta0), rosenbrock_hess(theta0)
eta_star = (g @ g) / (g @ Hm @ g)
etas = np.linspace(0, 2.5 * eta_star, 200)
true_vals = np.array([rosenbrock(theta0 - eta * g) for eta in etas])
first_order = rosenbrock(theta0) - etas * (g @ g)
second_order = first_order + 0.5 * etas ** 2 * (g @ Hm @ g)

plt.plot(etas, true_vals, label="true loss along $-\\nabla J$")
plt.plot(etas, first_order, "--", label="first-order (linear) prediction")
plt.plot(etas, second_order, "-.", label="second-order (quadratic) prediction")
plt.axvline(eta_star, color="gray", ls=":", label=f"$\\eta^* = ||g||^2 / (g^T H g)$ = {eta_star:.2e}")
plt.ylim(min(true_vals.min(), second_order.min()) - 1, rosenbrock(theta0) + 5)
plt.xlabel("learning rate $\\eta$"); plt.ylabel("loss after one step"); plt.title("Taylor expansion along the descent direction")
plt.legend(fontsize=8); plt.tight_layout(); plt.show()
print(f"loss at theta0: {rosenbrock(theta0):.3f} | after the step eta*: {rosenbrock(theta0 - eta_star * g):.3f} | "
      f"largest Hessian eigenvalue at theta0: {np.linalg.eigvalsh(Hm).max():.1f} -> safe eta < {1 / np.linalg.eigvalsh(Hm).max():.1e}")

Close to $\eta = 0$ all three curves agree, and the loss falls along the straight line with slope
$-\lVert \mathbf{g} \rVert^2$. The linear prediction keeps falling forever; the true loss does not.
The quadratic prediction captures the turn-around and its minimum $\eta^*$ is a good step, though
not perfect because the Rosenbrock function is not a quadratic. Beyond about $2\eta^*$ the step
increases the loss: gradient descent works because of the first-order term and is limited by the
second-order term.

### 3.2 Gradient descent on a quadratic, and the effect of the learning rate

On a quadratic bowl $J(\boldsymbol{\theta}) = \tfrac12 \boldsymbol{\theta}^\top \mathbf{A} \boldsymbol{\theta}$
with $\mathbf{A} = \mathrm{diag}(\lambda_1, \lambda_2)$, the gradient is $\mathbf{A}\boldsymbol{\theta}$ and
the update $\boldsymbol{\theta} \leftarrow \boldsymbol{\theta} - \eta \mathbf{A} \boldsymbol{\theta}$ acts on
each coordinate independently: $\theta_i \leftarrow (1 - \eta \lambda_i)\, \theta_i$. After $t$ steps
$\theta_i(t) = (1 - \eta\lambda_i)^t \theta_i(0)$, so coordinate $i$ converges if and only if
$|1 - \eta \lambda_i| < 1$, i.e. $0 < \eta < 2 / \lambda_i$, and the whole method converges iff
$\eta < 2 / \lambda_{\max}$, the bound from the previous section made exact. Four regimes follow:

- **too small** ($\eta \ll 1/\lambda_{\max}$): every factor is close to 1, progress is slow along every axis;
- **good** ($\eta \approx 1/\lambda_{\max}$): the stiffest direction is solved in one step, the others converge monotonically;
- **too large** ($1/\lambda_{\max} < \eta < 2/\lambda_{\max}$): the stiff coordinate flips sign at every step, converging while oscillating;
- **divergent** ($\eta > 2/\lambda_{\max}$): the stiff coordinate grows geometrically.

The cell implements gradient descent as a small function that records the path, runs it with
$\mathbf{A} = \mathrm{diag}(1, 10)$ from the same start for the four learning rates
$0.02$, $0.10$, $0.19$ and $0.21$ (the threshold is $2/10 = 0.2$), and plots the paths on the
contours. The optional `stop_above` argument halts a diverging run before it overflows.

In [ ]:
def gradient_descent(grad_fn, theta0, lr, n_steps, loss_fn=None, stop_above=1e12):
    """Plain gradient descent; returns the path as an array of shape (n_steps + 1, d)."""
    theta = np.array(theta0, dtype=np.float64)
    path = [theta.copy()]
    for _ in range(n_steps):
        theta = theta - lr * grad_fn(theta)
        path.append(theta.copy())
        if loss_fn is not None and not np.isfinite(loss_fn(theta)) or np.abs(theta).max() > stop_above:
            break
    return np.array(path)

A_quad = np.diag([1.0, 10.0])
quad_loss = lambda th: 0.5 * th @ A_quad @ th
quad_grad = lambda th: A_quad @ th
start = np.array([-2.0, 1.0])
q1, q2 = np.meshgrid(np.linspace(-2.5, 2.5, 101), np.linspace(-2.0, 2.0, 101))
Zq = 0.5 * (A_quad[0, 0] * q1 ** 2 + A_quad[1, 1] * q2 ** 2)

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for lr, ax in zip([0.02, 0.10, 0.19, 0.21], axes.ravel()):
    path = gradient_descent(quad_grad, start, lr, n_steps=25)
    ax.contour(q1, q2, Zq, levels=np.logspace(-2, 1.5, 15), cmap="viridis", alpha=0.7)
    ax.plot(path[:, 0], path[:, 1], "o-", color="C3", ms=3, lw=1)
    ax.scatter(*start, color="k", zorder=3); ax.scatter(0, 0, marker="*", color="gold", s=150, zorder=3, edgecolors="k")
    factors = 1 - lr * np.diag(A_quad)
    ax.set_title(f"lr = {lr}: factors |1 - lr*lambda| = {np.abs(factors[0]):.2f}, {np.abs(factors[1]):.2f}; final loss {quad_loss(path[-1]):.2e}", fontsize=9)
    ax.set_xlim(-2.5, 2.5); ax.set_ylim(-2.0, 2.0); ax.set_xlabel(r"$\theta_1$"); ax.set_ylabel(r"$\theta_2$")
plt.suptitle(r"Gradient descent on $\frac{1}{2}(\theta_1^2 + 10\,\theta_2^2)$; the star is the minimum"); plt.tight_layout(); plt.show()

The four panels are the four regimes. With $\eta = 0.02$ the path is smooth but after 25 steps is
still far from the minimum along $\theta_1$ (factor $0.98$ per step). With $\eta = 0.1$ the steep
$\theta_2$ direction is solved in one step (factor $0$) and the path glides along the valley floor.
With $\eta = 0.19$ the path zigzags across the valley (factor $-0.9$ on $\theta_2$) while still
converging. With $\eta = 0.21$ the zigzag grows (factor $-1.1$) and the iterate leaves the frame:
divergence. Every regime is predicted exactly by $|1 - \eta\lambda_i|$, and the practical lesson is
that the largest curvature sets the learning rate while the smallest curvature sets the speed.

### 3.3 The Rosenbrock function: a curved valley

Real loss surfaces are not quadratics. The Rosenbrock function has a global minimum at $(1, 1)$ at
the bottom of a curved, extremely narrow valley: the curvature across the valley near the minimum is
about $1000$ while the curvature along it is about $0.4$. From the previous section we know what
that means: the learning rate must stay below $2/1000$ to avoid divergence, and with such a step the
progress along the valley, with factor $1 - 0.4 \eta \approx 0.9996$ per step, is glacial. The cell
runs gradient descent from $(-1, 1.5)$ with two safe learning rates and one unsafe one, draws the
paths on a logarithmic contour plot, and reports how far from the optimum each run ends after the
step budget.

In [ ]:
STEPS = 5_000 if FAST_RUN else 20_000
r1, r2 = np.meshgrid(np.linspace(-1.6, 1.6, 200), np.linspace(-0.6, 2.0, 200))
Zr = rosenbrock(np.stack([r1, r2]))

fig, (ax_path, ax_loss) = plt.subplots(1, 2, figsize=(13, 4.8))
ax_path.contour(r1, r2, np.log10(Zr + 1e-3), levels=25, cmap="viridis", alpha=0.7)
for lr, color in [(5e-4, "C0"), (1e-3, "C3"), (4e-3, "C1")]:
    path = gradient_descent(rosenbrock_grad, [-1.0, 1.5], lr, STEPS, loss_fn=rosenbrock, stop_above=1e3)
    losses = np.array([rosenbrock(p) for p in path])
    diverged = len(path) < STEPS + 1
    ax_path.plot(path[:, 0], path[:, 1], "-", color=color, lw=1.2, label=f"lr = {lr:g}" + (" (diverged)" if diverged else ""))
    ax_loss.semilogy(losses, color=color, label=f"lr = {lr:g}")
    end = path[-1]
    print(f"lr = {lr:<7g} steps run {len(path) - 1:>6} | final point {end.round(4)} | distance to (1, 1): {np.linalg.norm(end - 1):.4f} | loss {losses[-1]:.2e}")
ax_path.scatter(1, 1, marker="*", color="gold", s=200, edgecolors="k", zorder=3)
ax_path.set_xlabel("$x$"); ax_path.set_ylabel("$y$"); ax_path.set_title("Paths on the Rosenbrock function (log contours)"); ax_path.legend(fontsize=8)
ax_loss.set_xlabel("step"); ax_loss.set_ylabel("loss (log scale)"); ax_loss.set_title("Loss versus step"); ax_loss.legend(fontsize=8)
plt.tight_layout(); plt.show()

The safe runs drop into the valley within a few dozen steps and then creep along it, thousands of
steps for the last stretch, and the larger of the two safe learning rates is twice as fast, as the
factor analysis predicts. The unsafe learning rate blows up almost immediately: the very first steps
land on the steep valley wall, where the curvature exceeds $2/\eta$. Neither problem is fixed by
tuning $\eta$ alone: any $\eta$ that is stable across the valley is slow along it. That tension is
the subject of Section 3.5.

### 3.4 Convex and non-convex landscapes: local minima, saddle points, plateaus

A function is **convex** if the line segment between any two points on its graph lies above the
graph; equivalently, for twice-differentiable functions, if the Hessian is positive semi-definite
everywhere. Quadratic bowls, least squares and logistic regression are convex, and for convex
functions every local minimum is a global one, so gradient descent with a suitable learning rate
converges to the best possible parameters. The loss of a neural network with at least one hidden
layer is **not convex**: permuting the hidden units gives a different parameter vector with the same
loss, so there are many equivalent minima with non-convex regions in between, and in general there
can be *distinct* local minima, saddle points and flat plateaus.

A point where the gradient vanishes is a **critical point**, and the eigenvalues of the Hessian
classify it: all positive means a local minimum, all negative a local maximum, and mixed signs a
**saddle point**, a minimum along some directions and a maximum along others. Gradient descent is
attracted to saddles along the descending directions and repelled only along the ascending ones,
and the repulsion is slow if the negative curvature is small, so a saddle can stall the optimizer for
a long time. In high dimensions saddles vastly outnumber local minima (for a random critical point
to be a minimum, *all* of thousands of curvatures must be positive), and Dauphin et al. (2014)
argued that they, not local minima, are the main obstacle in training deep networks. **Plateaus**,
regions where the gradient is tiny without any critical point nearby, arise from saturating
activation functions such as the sigmoid and are the origin of the vanishing-gradient problem met
in Notebook 02 · Neural Networks from Scratch (NumPy), Notebook 04 · Training Deep Networks:
Optimization, Initialization, Normalization, Regularization and Notebook 08 · Sequence Models:
RNNs, LSTMs and GRUs. The cell shows all three phenomena in one figure.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

# (a) local minima in one dimension: two starts, two different destinations
f_1d = lambda x: x ** 4 - 3 * x ** 2 + x
df_1d = lambda x: 4 * x ** 3 - 6 * x + 1
xs = np.linspace(-2.2, 2.2, 400)
axes[0].plot(xs, f_1d(xs), color="k")
for x_start, color in [(-0.3, "C3"), (0.3, "C0")]:
    path = gradient_descent(lambda x: np.array([df_1d(x[0])]), [x_start], lr=0.05, n_steps=60)[:, 0]
    axes[0].plot(path, f_1d(path), "o-", color=color, ms=3, label=f"start x = {x_start}: ends at {path[-1]:.2f}, f = {f_1d(path[-1]):.2f}")
axes[0].set_title("(a) non-convex: where you end depends on where you start", fontsize=9); axes[0].legend(fontsize=8); axes[0].set_xlabel("$x$")

# (b) a saddle point: f = x^2 - y^2; the origin is a minimum along x and a maximum along y
s1, s2 = np.meshgrid(np.linspace(-1.5, 1.5, 100), np.linspace(-1.5, 1.5, 100))
axes[1].contour(s1, s2, s1 ** 2 - s2 ** 2, levels=20, cmap="coolwarm")
saddle_grad = lambda p: np.array([2 * p[0], -2 * p[1]])
for y_start, color, label in [(0.0, "C3", "start exactly on the ridge: stuck at the saddle"), (1e-3, "C0", "start perturbed by 0.001: escapes")]:
    path = gradient_descent(saddle_grad, [1.4, y_start], lr=0.1, n_steps=40, stop_above=1.5)
    axes[1].plot(path[:, 0], path[:, 1], "o-", color=color, ms=3, lw=1, label=label)
axes[1].set_title("(b) saddle point of $x^2 - y^2$; Hessian eigenvalues +2 and -2", fontsize=9); axes[1].legend(fontsize=7)
axes[1].set_xlim(-1.5, 1.5); axes[1].set_ylim(-1.5, 1.5)

# (c) a plateau: a saturated sigmoid-shaped loss; the gradient is tiny far from the transition
sigmoid = lambda x: 1 / (1 + np.exp(-x))
f_plateau = lambda x: 1 - sigmoid(x)
df_plateau = lambda x: -sigmoid(x) * (1 - sigmoid(x))
xs = np.linspace(-12, 8, 400)
axes[2].plot(xs, f_plateau(xs), color="k")
path = gradient_descent(lambda x: np.array([df_plateau(x[0])]), [-8.0], lr=5.0, n_steps=60)[:, 0]
axes[2].plot(path, f_plateau(path), "o-", color="C3", ms=3, lw=1, label=f"60 steps at lr = 5 from x = -8: reached x = {path[-1]:.2f}")
axes[2].set_title("(c) plateau: gradient $\\approx$ 0 although we are far from the minimum", fontsize=9); axes[2].legend(fontsize=8); axes[2].set_xlabel("$x$")
plt.tight_layout(); plt.show()
print(f"gradient magnitude on the plateau at x = -8: {abs(df_plateau(-8.0)):.2e}  (versus 0.25 at x = 0)")

In (a) two starting points a short distance apart end in different valleys with different losses;
the optimizer has no way of knowing that the deeper one exists. In (b) a start exactly on the ridge
converges to the saddle and stays there, since the gradient is zero, while a perturbation of one part
in a thousand is enough to escape, slowly at first and then quickly, along the direction of negative
curvature; the noise of mini-batch gradients provides such perturbations for free, one reason
stochastic gradient descent works better than its deterministic counterpart. In (c) the gradient at
$x = -8$ is about $3 \times 10^{-4}$, so even sixty steps at a large learning rate barely move; a
network whose sigmoid units are saturated is stuck on exactly such a plateau.

### 3.5 Conditioning: elongated valleys and why we need momentum

Return to the quadratic. The ratio of the largest to the smallest curvature,
$\kappa = \lambda_{\max} / \lambda_{\min}$, is the **condition number** of the Hessian (compare
Section 1.5), and it governs the speed of gradient descent. With the step
$\eta = 1 / \lambda_{\max}$ (the largest for which no coordinate oscillates, Section 3.2), the
slowest coordinate shrinks by the factor
$1 - \lambda_{\min}/\lambda_{\max} = 1 - 1/\kappa$ per step, so reaching a relative accuracy
$\epsilon$ takes about $\kappa \log(1/\epsilon)$ steps. A condition number of $10^4$, common for real
losses, means tens of thousands of steps. Ill-conditioning looks like an elongated valley: steep
walls that force a small step, and a long, gently sloping floor that the small step crawls along.

**Momentum** attacks this directly. Instead of stepping along the current gradient, it accumulates
a velocity,

$$
\mathbf{v} \leftarrow \beta \mathbf{v} + \nabla J(\boldsymbol{\theta}), \qquad
\boldsymbol{\theta} \leftarrow \boldsymbol{\theta} - \eta \mathbf{v},
$$

with $\beta \approx 0.9$. Along the valley floor, where successive gradients point the same way, the
velocity builds up to $1/(1 - \beta) = 10$ times the gradient, an effective ten-fold learning rate;
across the valley, where successive gradients alternate in sign, they cancel in the velocity and
the zigzag is damped. The theory (Polyak, 1964) shows that the number of steps drops from
$O(\kappa)$ to $O(\sqrt{\kappa})$, a hundred-fold improvement for $\kappa = 10^4$. Adaptive
methods such as RMSProp and Adam rescale each coordinate by its own gradient history, which
approximately equalises the curvatures instead; Notebook 04 derives them all and compares them on
real networks.
The cell compares plain gradient descent with momentum on a quadratic with $\kappa = 50$, at the
same learning rate. (For the record: eliminating the velocity gives the two-step recurrence
$\theta_{t+1} = (1 + \beta - \eta\lambda_i)\theta_t - \beta\theta_{t-1}$ per coordinate, whose
characteristic roots are complex, with modulus exactly $\sqrt{\beta}$, whenever
$(1 - \sqrt{\beta})^2 < \eta\lambda_i < (1 + \sqrt{\beta})^2$; for $\beta = 0.9$ that is the wide
band $0.003 < \eta\lambda_i < 3.8$. Every direction in the band contracts at the same rate
$\sqrt{\beta}$ per step, independent of its curvature, which is why a single $\beta$ helps all of
them at once.)

In [ ]:
def momentum_descent(grad_fn, theta0, lr, beta, n_steps):
    """Heavy-ball momentum; returns the path with shape (n_steps + 1, d)."""
    theta, v = np.array(theta0, dtype=np.float64), np.zeros(len(theta0))
    path = [theta.copy()]
    for _ in range(n_steps):
        v = beta * v + grad_fn(theta)
        theta = theta - lr * v
        path.append(theta.copy())
    return np.array(path)

kappa = 50.0
A_ill = np.diag([1.0, kappa])
ill_loss = lambda th: 0.5 * th @ A_ill @ th
ill_grad = lambda th: A_ill @ th
start = np.array([-2.0, 0.5])
N_STEPS = 300
runs = {
    "GD, lr = 1/lambda_max = 0.02": gradient_descent(ill_grad, start, 1 / kappa, N_STEPS),
    "GD, optimal lr = 2/(l_max + l_min) = 0.039": gradient_descent(ill_grad, start, 2 / (kappa + 1), N_STEPS),
    "momentum, lr = 0.02, beta = 0.9": momentum_descent(ill_grad, start, 1 / kappa, 0.9, N_STEPS),
}
c1, c2 = np.meshgrid(np.linspace(-2.5, 2.5, 101), np.linspace(-0.8, 0.8, 101))
Zc = 0.5 * (c1 ** 2 + kappa * c2 ** 2)
fig, (ax_p, ax_l) = plt.subplots(1, 2, figsize=(13, 4.6))
ax_p.contour(c1, c2, Zc, levels=np.logspace(-3, 1.3, 14), cmap="viridis", alpha=0.6)
for (label, path), color in zip(runs.items(), ["C3", "C1", "C0"]):
    ax_p.plot(path[:40, 0], path[:40, 1], "o-", color=color, ms=2.5, lw=1, label=label + " (first 40 steps)")
    ax_l.semilogy([ill_loss(p) for p in path], color=color, label=label)
    print(f"{label:<45} loss after {N_STEPS} steps: {ill_loss(path[-1]):.2e}")
ax_p.scatter(0, 0, marker="*", color="gold", s=150, edgecolors="k", zorder=3)
ax_p.set_xlabel(r"$\theta_1$"); ax_p.set_ylabel(r"$\theta_2$"); ax_p.set_title(f"Ill-conditioned quadratic, condition number {kappa:.0f}"); ax_p.legend(fontsize=7)
ax_l.set_xlabel("step"); ax_l.set_ylabel("loss"); ax_l.set_title("Convergence"); ax_l.legend(fontsize=7)
plt.tight_layout(); plt.show()
print(f"theory: GD at lr = 1/lambda_max needs about kappa = {kappa:.0f} steps per e-fold; optimally tuned momentum about sqrt(kappa) = {np.sqrt(kappa):.1f};")
print(f"        here, with beta = 0.9, the error contracts by sqrt(beta) = {np.sqrt(0.9):.3f} per step, i.e. one e-fold every {1 / -np.log(np.sqrt(0.9)):.0f} steps")

At the same learning rate, momentum contracts the error by a factor of about $\sqrt{0.9} \approx 0.95$
per step instead of $0.98$: it reaches in roughly a hundred steps the loss that plain gradient
descent reaches only after three hundred, and after three hundred steps it is about three orders of
magnitude ahead of even the theoretically optimal fixed learning rate for gradient descent. In the
path plot, momentum oscillates across the valley for a while (the price of carrying velocity) while
advancing along the floor far faster than gradient descent, which inches forward. Real losses are
far worse conditioned than $\kappa = 50$, which is why no one trains a deep network with plain
gradient descent.

> **Try it.** Set `kappa = 500` and rerun the cell. Plain gradient descent at $\eta = 1/\lambda_{\max}$
> now covers only about half of the distance along $\theta_1$ in three hundred steps (factor
> $0.998$ per step), while momentum with $\beta = 0.9$ still drives the loss below $10^{-6}$; then
> try $\beta = 0.99$ and watch the oscillations across the valley take much longer to die out.

> **Key idea.** Gradient descent decreases the loss because of the first-order Taylor term, is
> limited in step size by the largest curvature, and is slowed down by the smallest; the ratio of the
> two, the condition number, is what momentum and adaptive optimizers are designed to tame.

## 4. Probability and information theory

Why does a classifier minimise cross-entropy and a regressor mean-squared error? Not because
somebody chose them: both fall out of a single principle, **maximum likelihood**, applied to a
particular assumption about the noise in the data. This section builds the vocabulary needed to state
that principle and then carries out the two derivations. Along the way we meet entropy and the KL
divergence, which are the language of Notebook 11 · Autoencoders and Variational Autoencoders and
Notebook 12 · GANs and Diffusion Models, and end with the softmax and the log-sum-exp trick that
every classifier in the course relies on.

### 4.1 Random variables, expectation and variance

A **random variable** $X$ is a quantity whose value is determined by chance, described by a
distribution: a **probability mass function** $p(x) = P(X = x)$ for discrete outcomes, or a
**probability density** $p(x)$ for continuous ones, with $P(a \le X \le b) = \int_a^b p(x)\,dx$.
The **expectation** is the probability-weighted average of the outcomes,

$$
\mathbb{E}[X] = \sum_x x\, p(x) \quad \text{or} \quad \int x\, p(x)\, dx,
$$

and, more generally, $\mathbb{E}[g(X)] = \sum_x g(x) p(x)$ for any function $g$. Expectation is
**linear**: $\mathbb{E}[aX + bY] = a\mathbb{E}[X] + b\mathbb{E}[Y]$ always, whether or not $X$ and $Y$
are related. The **variance** measures spread around the mean,

$$
\mathrm{Var}[X] = \mathbb{E}\big[(X - \mathbb{E}[X])^2\big] = \mathbb{E}[X^2] - \mathbb{E}[X]^2,
$$

and its square root is the **standard deviation**; scaling gives $\mathrm{Var}[aX] = a^2 \mathrm{Var}[X]$.
When the distribution is unknown or the sum is intractable, we estimate expectations by averaging
samples, a **Monte Carlo** estimate, which converges to the true value as the number of samples
grows (the **law of large numbers**). Almost every quantity in deep learning is an expectation
estimated this way: the training loss is an average over a mini-batch, a gradient is an average of
per-example gradients, and a policy's return in Notebook 13 · Reinforcement Learning is an average
over episodes.

The cell computes the exact mean and variance of a fair die, checks the $\mathbb{E}[X^2] - \mathbb{E}[X]^2$
identity, and shows the running Monte Carlo estimate of the mean converging.

In [ ]:
faces = np.arange(1, 7)
probs = np.full(6, 1 / 6)
mean_exact = faces @ probs
var_exact = ((faces - mean_exact) ** 2) @ probs
print(f"fair die: E[X] = {mean_exact:.4f} | Var[X] = {var_exact:.4f} | E[X^2] - E[X]^2 = {(faces ** 2) @ probs - mean_exact ** 2:.4f}")

n_samples = 100_000
samples = rng.integers(1, 7, size=n_samples)
running_mean = np.cumsum(samples) / np.arange(1, n_samples + 1)
print(f"Monte Carlo with {n_samples:,} samples: mean {samples.mean():.4f}, variance {samples.var():.4f}")

plt.semilogx(np.arange(1, n_samples + 1), running_mean, lw=1, label="running sample mean")
plt.axhline(mean_exact, color="k", ls="--", label="exact expectation 3.5")
plt.fill_between(np.arange(1, n_samples + 1), mean_exact - 2 * np.sqrt(var_exact / np.arange(1, n_samples + 1)),
                 mean_exact + 2 * np.sqrt(var_exact / np.arange(1, n_samples + 1)), color="gray", alpha=0.2, label=r"$\pm 2\sigma/\sqrt{n}$")
plt.xlabel("number of samples $n$ (log scale)"); plt.ylabel("estimate of E[X]"); plt.title("Law of large numbers for a fair die")
plt.legend(); plt.tight_layout(); plt.show()

The running mean wanders early on and then settles inside the shrinking band $\pm 2\sigma / \sqrt{n}$:
the error of a Monte Carlo estimate decreases like $1/\sqrt{n}$, so ten times more samples buy only
about three times more accuracy. This is exactly why mini-batch gradients are noisy and why larger
batches reduce (but only slowly) that noise, a trade-off explored in Notebook 04 · Training Deep
Networks: Optimization, Initialization, Normalization, Regularization.

### 4.2 Three distributions: Bernoulli, categorical, Gaussian

Three distributions cover nearly all of the outputs a deep network is asked to model.

The **Bernoulli** distribution describes a binary outcome $x \in \{0, 1\}$ with
$p(x) = \phi^x (1 - \phi)^{1 - x}$: probability $\phi$ of a one, mean $\phi$, variance $\phi(1 - \phi)$.
It is the output distribution of a binary classifier, whose network produces $\phi = \sigma(z)$ from a
logit $z$.

The **categorical** distribution generalises it to $K$ outcomes, $p(x = k) = \pi_k$ with
$\pi_k \ge 0$ and $\sum_k \pi_k = 1$. A $K$-class classifier outputs $\boldsymbol{\pi} = \mathrm{softmax}(\mathbf{z})$,
and a language model outputs a categorical distribution over its vocabulary at every position.

The **Gaussian** (normal) distribution $\mathcal{N}(x; \mu, \sigma^2)$ has the density

$$
p(x) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\Big( -\frac{(x - \mu)^2}{2\sigma^2} \Big),
$$

with mean $\mu$ and variance $\sigma^2$. It is the default model for continuous noise for three
reasons: the **central limit theorem** says sums of many small independent effects are approximately
Gaussian; among all distributions with a given mean and variance it has the maximum entropy (it
assumes the least); and it is closed under linear operations, which makes the mathematics of
Notebooks 11 and 12 tractable. Its multivariate version $\mathcal{N}(\mathbf{x}; \boldsymbol{\mu}, \boldsymbol{\Sigma})$
replaces $\sigma^2$ by a covariance matrix. The cell plots all three, together with histograms of
samples drawn with NumPy's generator to confirm that the samples follow the formulas.

In [ ]:
def gaussian_pdf(x, mu, sigma):
    return np.exp(-(x - mu) ** 2 / (2 * sigma ** 2)) / np.sqrt(2 * np.pi * sigma ** 2)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

phi = 0.3
bern_samples = rng.random(10_000) < phi
axes[0].bar([0, 1], [1 - phi, phi], width=0.5, alpha=0.6, label=r"pmf, $\phi = 0.3$")
axes[0].plot([0, 1], [1 - bern_samples.mean(), bern_samples.mean()], "ko", label="frequency in 10,000 samples")
axes[0].set_xticks([0, 1]); axes[0].set_title(f"Bernoulli: mean {bern_samples.mean():.3f}, variance {bern_samples.var():.3f} (exact {phi * (1 - phi):.3f})", fontsize=9); axes[0].legend(fontsize=8)

logits = rng.standard_normal(6)
pi = softmax(logits)
cat_samples = rng.choice(6, size=10_000, p=pi)
axes[1].bar(np.arange(6), pi, alpha=0.6, label="pmf = softmax(logits)")
axes[1].plot(np.arange(6), np.bincount(cat_samples, minlength=6) / len(cat_samples), "ko", label="frequency in samples")
axes[1].set_xlabel("class $k$"); axes[1].set_title("Categorical over 6 classes", fontsize=9); axes[1].legend(fontsize=8)

xs = np.linspace(-6, 6, 400)
for sigma in [0.5, 1.0, 2.0]:
    axes[2].plot(xs, gaussian_pdf(xs, 0.0, sigma), label=rf"$\sigma = {sigma}$")
axes[2].hist(rng.normal(0.0, 1.0, 10_000), bins=60, density=True, alpha=0.3, color="C1", label=r"samples, $\sigma = 1$")
axes[2].set_xlabel("$x$"); axes[2].set_title(r"Gaussian densities $\mathcal{N}(0, \sigma^2)$", fontsize=9); axes[2].legend(fontsize=8)
plt.tight_layout(); plt.show()

The sample frequencies sit on the theoretical bars and the histogram follows the curve, as they
should. Note the Gaussian's shape: the density falls off as $e^{-x^2/2\sigma^2}$, so values five
standard deviations from the mean are practically impossible; this "thin tail" is the reason
squared-error losses, which are derived from it in Section 4.4, are so sensitive to outliers.

### 4.3 Joint, conditional and Bayes' rule

Two random variables $X$ and $Y$ have a **joint** distribution $p(x, y)$. Summing out one variable
gives the **marginal** of the other, $p(x) = \sum_y p(x, y)$, and dividing gives the **conditional**
$p(y \mid x) = p(x, y) / p(x)$, the distribution of $Y$ once $X$ is known. Rearranged, this is the
**product rule** $p(x, y) = p(y \mid x)\, p(x) = p(x \mid y)\, p(y)$, and equating the two
factorisations gives **Bayes' rule**:

$$
p(y \mid x) = \frac{p(x \mid y)\, p(y)}{p(x)}, \qquad p(x) = \sum_{y} p(x \mid y)\, p(y).
$$

$X$ and $Y$ are **independent** if $p(x, y) = p(x)p(y)$, i.e. knowing one tells you nothing about the
other. A supervised model is a conditional distribution $p(y \mid \mathbf{x})$; a generative model
(Part V) models $p(\mathbf{x})$ or the joint; and Bayes' rule is how a probabilistic classifier turns
class-conditional densities into posterior class probabilities.

The classic example makes the rule concrete. A disease has prevalence $p(D) = 0.01$; a test detects
it with probability $p(+ \mid D) = 0.9$ (sensitivity) and gives a false positive with probability
$p(+ \mid \neg D) = 0.05$. What is $p(D \mid +)$? The cell builds the $2 \times 2$ joint table from the
prior and the likelihoods, marginalises and conditions with array operations, and confirms the answer
by simulation.

In [ ]:
prior = np.array([0.99, 0.01])                          # p(no disease), p(disease)
likelihood = np.array([[0.95, 0.05],                    # rows: disease state; columns: test negative, positive
                       [0.10, 0.90]])
joint = prior[:, None] * likelihood                      # p(d, t) = p(t | d) p(d), shape (2, 2)
p_test = joint.sum(axis=0)                               # marginal over the disease state -> p(t)
posterior = joint / p_test                               # p(d | t) = p(d, t) / p(t), column-wise
print("joint p(disease, test):\n", joint, "\n(sums to", joint.sum().round(6), ")")
print("p(test positive) =", p_test[1].round(4))
print("p(disease | positive) =", posterior[1, 1].round(4), "   p(disease | negative) =", posterior[1, 0].round(6))
print("Bayes' rule directly: p(+|D) p(D) / p(+) =", (likelihood[1, 1] * prior[1] / p_test[1]).round(4))

n = 1_000_000
disease = rng.random(n) < prior[1]
positive = np.where(disease, rng.random(n) < likelihood[1, 1], rng.random(n) < likelihood[0, 1])
print(f"simulation with {n:,} people: fraction with the disease among positives = {disease[positive].mean():.4f}")

A positive result from a 90%-sensitive, 95%-specific test means only a 15% chance of disease,
because the disease is rare and the 5% false positives among the healthy majority outnumber the true
positives. The simulation agrees to within a few tenths of a percent (the Monte Carlo error of a
proportion estimated from about 58,000 positives). The same arithmetic appears whenever a classifier
is evaluated on imbalanced data, and in Notebook 16 · Interpretability, Robustness and Uncertainty
when we ask how much to trust a confident prediction.

### 4.4 Maximum likelihood: where MSE and cross-entropy come from

Suppose we have a family of models $p(y \mid \mathbf{x}; \boldsymbol{\theta})$ and a dataset of
independent examples $\{(\mathbf{x}^{(i)}, y^{(i)})\}_{i=1}^N$. The **likelihood** of the parameters is
the probability the model assigns to the data,

$$
\mathcal{L}(\boldsymbol{\theta}) = \prod_{i=1}^{N} p\big(y^{(i)} \mid \mathbf{x}^{(i)}; \boldsymbol{\theta}\big),
$$

and the **maximum likelihood estimate** chooses the $\boldsymbol{\theta}$ that makes the observed data
most probable. Because a product of many small numbers is numerically hopeless (Section 5.2) and
because sums are easier to differentiate, we maximise the **log-likelihood** instead, or equivalently
minimise the **negative log-likelihood** (NLL)

$$
\mathrm{NLL}(\boldsymbol{\theta}) = -\sum_{i=1}^{N} \log p\big(y^{(i)} \mid \mathbf{x}^{(i)}; \boldsymbol{\theta}\big).
$$

Every standard loss function is an NLL under some model of the data. Two cases cover most of the
course.

**Gaussian noise gives mean-squared error.** Assume $y = f_{\boldsymbol{\theta}}(\mathbf{x}) + \epsilon$
with $\epsilon \sim \mathcal{N}(0, \sigma^2)$, i.e. $p(y \mid \mathbf{x}; \boldsymbol{\theta}) =
\mathcal{N}(y; f_{\boldsymbol{\theta}}(\mathbf{x}), \sigma^2)$. Taking the log of the Gaussian density,

$$
-\log p(y \mid \mathbf{x}; \boldsymbol{\theta}) = \tfrac12 \log(2\pi\sigma^2) + \frac{\big(y - f_{\boldsymbol{\theta}}(\mathbf{x})\big)^2}{2\sigma^2},
\qquad
\mathrm{NLL}(\boldsymbol{\theta}) = \frac{N}{2}\log(2\pi\sigma^2) + \frac{1}{2\sigma^2} \sum_{i=1}^{N} \big(y^{(i)} - f_{\boldsymbol{\theta}}(\mathbf{x}^{(i)})\big)^2 .
$$

The first term does not depend on $\boldsymbol{\theta}$ and the factor $1/2\sigma^2$ is a positive
constant, so minimising the NLL over $\boldsymbol{\theta}$ is *exactly* minimising the sum of squared
errors, i.e. the **mean-squared error**. The squared error is not an arbitrary choice: it is the
assumption that the noise is Gaussian. (Assume Laplacian noise instead and the same derivation gives
the absolute error.)

**Bernoulli and categorical models give cross-entropy.** For binary labels, let the network output a
probability $\hat{p} = \sigma(f_{\boldsymbol{\theta}}(\mathbf{x}))$ and model $y \sim \mathrm{Bernoulli}(\hat{p})$.
Then $p(y \mid \mathbf{x}) = \hat{p}^{\,y} (1 - \hat{p})^{1 - y}$ and

$$
\mathrm{NLL}(\boldsymbol{\theta}) = -\sum_{i=1}^{N} \Big[ y^{(i)} \log \hat{p}^{(i)} + (1 - y^{(i)}) \log (1 - \hat{p}^{(i)}) \Big],
$$

which is the **binary cross-entropy** that `nn.BCEWithLogitsLoss` computes from the logit. For $K$
classes with $\hat{\mathbf{p}} = \mathrm{softmax}(f_{\boldsymbol{\theta}}(\mathbf{x}))$ and
$y \sim \mathrm{Categorical}(\hat{\mathbf{p}})$, the probability of the observed label is $\hat{p}_{y}$ and

$$
\mathrm{NLL}(\boldsymbol{\theta}) = -\sum_{i=1}^{N} \log \hat{p}^{(i)}_{y^{(i)}},
$$

the **cross-entropy loss** (`nn.CrossEntropyLoss`). Minimising it pushes up the probability of the
correct class, and, because of the $\log$, a confident wrong prediction ($\hat p_y \to 0$) is
punished without bound, which is the source of its strong gradients (Notebook 02 · Neural Networks
from Scratch (NumPy) derives the gradient $\hat{\mathbf{p}} - \mathbf{y}$).

The cell makes the principle tangible with the simplest models. For Gaussian data it computes the
NLL as a function of $(\mu, \sigma)$, shows that its minimum is at the sample mean and the sample
standard deviation (the closed-form MLE), and recovers the same values by gradient descent on the
NLL, started from $(\mu, \sigma) = (1, 1)$ and parameterised by $\log \sigma$ so that the standard
deviation can never become negative; for coin flips it shows that the Bernoulli NLL is minimised at
the observed frequency.

In [ ]:
data_g = rng.normal(loc=2.0, scale=1.5, size=500)

def gaussian_nll(mu, sigma, x):
    return 0.5 * len(x) * np.log(2 * np.pi * sigma ** 2) + np.sum((x - mu) ** 2) / (2 * sigma ** 2)

mu_hat, sigma_hat = data_g.mean(), np.sqrt(np.mean((data_g - data_g.mean()) ** 2))    # closed-form MLE (note: N, not N-1)

# gradient descent on the mean NLL with parameters (mu, s = log sigma) so that sigma stays positive
def mean_nll(params):
    mu, s = params
    return gaussian_nll(mu, np.exp(s), data_g) / len(data_g)

def mean_nll_grad(params):
    mu, s = params
    sigma = np.exp(s)
    return np.array([-np.mean(data_g - mu) / sigma ** 2, 1 - np.mean((data_g - mu) ** 2) / sigma ** 2])

gradient_check(mean_nll, mean_nll_grad, [1.0, 0.2], name="Gaussian NLL gradient")
path = gradient_descent(mean_nll_grad, [1.0, 0.0], lr=0.5, n_steps=200)     # start at mu = 1, log sigma = 0
mu_gd, sigma_gd = path[-1, 0], np.exp(path[-1, 1])
print(f"closed-form MLE: mu = {mu_hat:.4f}, sigma = {sigma_hat:.4f} | gradient descent: mu = {mu_gd:.4f}, sigma = {sigma_gd:.4f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.3))
mu_grid, sig_grid = np.meshgrid(np.linspace(1.0, 3.0, 101), np.linspace(0.8, 3.4, 101))   # the first step overshoots to sigma ~ 3.2
nll_grid = np.array([[gaussian_nll(m, s, data_g) for m in mu_grid[0]] for s in sig_grid[:, 0]])
ax1.contour(mu_grid, sig_grid, nll_grid, levels=30, cmap="viridis")
ax1.plot(path[:, 0], np.exp(path[:, 1]), "r.-", ms=3, lw=1, label="gradient descent path")
ax1.scatter(mu_hat, sigma_hat, marker="*", s=200, color="gold", edgecolors="k", zorder=3, label="closed-form MLE")
ax1.set_xlim(1.0, 3.0); ax1.set_ylim(0.8, 3.4); ax1.set_xlabel(r"$\mu$"); ax1.set_ylabel(r"$\sigma$"); ax1.set_title("Gaussian NLL of 500 samples"); ax1.legend(fontsize=8)

flips = (rng.random(200) < 0.3).astype(float)
phis = np.linspace(0.005, 0.995, 400)
bern_nll = -np.array([np.sum(flips * np.log(ph) + (1 - flips) * np.log(1 - ph)) for ph in phis])
ax2.plot(phis, bern_nll); ax2.axvline(flips.mean(), color="k", ls="--", label=f"observed frequency {flips.mean():.3f}")
ax2.scatter(phis[np.argmin(bern_nll)], bern_nll.min(), color="gold", edgecolors="k", s=150, marker="*", zorder=3, label=f"minimum at {phis[np.argmin(bern_nll)]:.3f}")
ax2.set_xlabel(r"$\phi$"); ax2.set_ylabel("negative log-likelihood"); ax2.set_title("Bernoulli NLL of 200 coin flips"); ax2.legend(fontsize=8)
plt.tight_layout(); plt.show()

Gradient descent on the NLL lands on the same $(\mu, \sigma)$ as the closed-form estimate, and the
Bernoulli NLL bottoms out at the empirical frequency of heads. These are the two simplest
instances of a pattern that recurs in every later notebook: *write down a probabilistic model of the
outputs, take the negative log-likelihood, and minimise it by gradient descent*. When the model's
mean is a neural network instead of a constant, nothing changes except the size of $\boldsymbol{\theta}$.

### 4.5 Entropy, cross-entropy and KL divergence

Information theory quantifies how surprising outcomes are. The **entropy** of a distribution $p$ over
$K$ outcomes,

$$
H(p) = -\sum_{k=1}^{K} p_k \log p_k = \mathbb{E}_{x \sim p}[-\log p(x)],
$$

is the expected surprise ($-\log p(x)$ is large for improbable events) and measures how uncertain
the outcome is: zero for a deterministic outcome, maximal ($\log K$) for the uniform distribution.
With the natural logarithm the unit is the *nat*; with $\log_2$ it is the *bit*. The
**cross-entropy** of $q$ relative to $p$,

$$
H(p, q) = -\sum_k p_k \log q_k,
$$

is the expected surprise if outcomes follow $p$ but we *believe* $q$; it is at least $H(p)$, with
equality only when $q = p$. Their difference is the **Kullback–Leibler divergence**

$$
D_{\mathrm{KL}}(p \,\|\, q) = \sum_k p_k \log \frac{p_k}{q_k} = H(p, q) - H(p) \;\ge\; 0,
$$

the extra surprise caused by using the wrong distribution. Non-negativity (Gibbs' inequality)
follows from $\log t \le t - 1$: $-D_{\mathrm{KL}} = \sum_k p_k \log(q_k / p_k) \le \sum_k p_k (q_k/p_k - 1) = \sum_k q_k - \sum_k p_k = 0$.
The KL divergence is not symmetric, $D_{\mathrm{KL}}(p\|q) \ne D_{\mathrm{KL}}(q\|p)$ in general, and the
two directions penalise different mistakes: $D_{\mathrm{KL}}(p\|q)$ explodes when $q$ assigns
near-zero probability to something $p$ considers possible, whereas $D_{\mathrm{KL}}(q\|p)$ is
forgiving of that but punishes $q$ for putting mass where $p$ has none. This asymmetry decides which
direction the variational objective of Notebook 11 uses.

The connection to training: if $p$ is the one-hot distribution of the true label and $q = \hat{\mathbf{p}}$
the model's prediction, then $H(p, q) = -\log \hat{p}_y$, exactly the cross-entropy loss of Section
4.4. And since $H(p)$ does not depend on the model, minimising cross-entropy is the same as
minimising $D_{\mathrm{KL}}(p_{\text{data}} \,\|\, p_{\text{model}})$: training a classifier by maximum
likelihood means making the model distribution as close as possible, in KL, to the data distribution.
The cell implements the three quantities and works through numeric examples.

In [ ]:
def entropy(p):
    p = np.asarray(p, dtype=np.float64); nz = p > 0
    return -np.sum(p[nz] * np.log(p[nz])) + 0.0              # 0 log 0 is taken to be 0; "+ 0.0" turns -0.0 into 0.0

def cross_entropy(p, q):
    p, q = np.asarray(p, dtype=np.float64), np.asarray(q, dtype=np.float64); nz = p > 0
    return -np.sum(p[nz] * np.log(q[nz]))

def kl_divergence(p, q):
    return cross_entropy(p, q) - entropy(p)

print(f"entropy of a fair coin      : {entropy([0.5, 0.5]):.4f} nats = {entropy([0.5, 0.5]) / np.log(2):.3f} bits")
print(f"entropy of a 90/10 coin     : {entropy([0.9, 0.1]):.4f} nats")
print(f"entropy of a certain outcome: {entropy([1.0, 0.0]):.4f} nats")
print(f"entropy of uniform over 10  : {entropy(np.full(10, 0.1)):.4f} nats = log(10) = {np.log(10):.4f}")

p_true, q_model = np.array([0.7, 0.2, 0.1]), np.array([0.5, 0.3, 0.2])
print(f"\nH(p) = {entropy(p_true):.4f} | H(p, q) = {cross_entropy(p_true, q_model):.4f} | "
      f"KL(p||q) = {kl_divergence(p_true, q_model):.4f} | KL(q||p) = {kl_divergence(q_model, p_true):.4f}  (not symmetric)")
one_hot = np.array([0.0, 1.0, 0.0])
print(f"cross-entropy of a one-hot label against q: {cross_entropy(one_hot, q_model):.4f} = -log q_y = {-np.log(q_model[1]):.4f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
phis = np.linspace(0.001, 0.999, 300)
ax1.plot(phis, [entropy([ph, 1 - ph]) for ph in phis]); ax1.set_xlabel(r"$\phi$"); ax1.set_ylabel("entropy (nats)")
ax1.set_title(r"Entropy of Bernoulli($\phi$): maximal at $\phi = 0.5$")
p_fixed = 0.2
ax2.plot(phis, [kl_divergence([p_fixed, 1 - p_fixed], [ph, 1 - ph]) for ph in phis], label=r"$D_{KL}(p \| q)$")
ax2.plot(phis, [kl_divergence([ph, 1 - ph], [p_fixed, 1 - p_fixed]) for ph in phis], "--", label=r"$D_{KL}(q \| p)$")
ax2.axvline(p_fixed, color="gray", ls=":", label=r"$q = p$ (both zero)"); ax2.set_ylim(0, 3)
ax2.set_xlabel(r"parameter of $q$"); ax2.set_ylabel("divergence (nats)"); ax2.set_title(r"KL between Bernoulli(0.2) and Bernoulli($\cdot$) is asymmetric"); ax2.legend(fontsize=8)
plt.tight_layout(); plt.show()

The numbers match the formulas: a fair coin carries one bit, a certain outcome carries nothing, and
the uniform distribution over ten classes carries $\log 10$ nats, which is why the cross-entropy loss
of an untrained ten-class classifier starts near $2.30$, a useful sanity check that Notebook 04
turns into a diagnostic. The right-hand plot shows the asymmetry: both divergences vanish only at
$q = p$, but they grow at different rates on either side.

### 4.6 Softmax, its derivative, and the log-sum-exp trick

The **softmax** function turns a vector of real-valued scores (logits) into a categorical
distribution:

$$
\mathrm{softmax}(\mathbf{z})_i = \frac{e^{z_i}}{\sum_{k=1}^{K} e^{z_k}} .
$$

It is positive, sums to one, is monotone in each logit, and approaches a one-hot vector on the
largest logit as the differences between logits grow (hence "soft" maximum). Its Jacobian,
$\partial p_i / \partial z_j = p_i(\delta_{ij} - p_j)$, was derived in Section 2.5. A property we
now need is **shift invariance**: adding the same constant $c$ to every logit changes nothing,
because $e^{z_i + c} / \sum_k e^{z_k + c} = e^{c} e^{z_i} / (e^{c} \sum_k e^{z_k})$ and the
$e^{c}$ cancels.

This matters because $e^{z}$ overflows single precision for $z > 88$ and double precision for
$z > 709$, and logits of that size occur routinely during training. The cure is to subtract the
maximum logit before exponentiating: $\mathrm{softmax}(\mathbf{z}) = \mathrm{softmax}(\mathbf{z} - \max_k z_k)$,
after which the largest exponent is $e^0 = 1$ and nothing can overflow (small entries may underflow
to zero, which is harmless in the sum because the maximum term is exactly one).

For the loss we need $\log$ of the softmax, and naively computing $\log(\mathrm{softmax}(\mathbf{z}))$
fails in a second way: a probability that underflows to zero gives $\log 0 = -\infty$. Instead use

$$
\log \mathrm{softmax}(\mathbf{z})_i = z_i - \mathrm{LSE}(\mathbf{z}), \qquad
\mathrm{LSE}(\mathbf{z}) = \log \sum_k e^{z_k} = c + \log \sum_k e^{z_k - c}, \quad c = \max_k z_k .
$$

The **log-sum-exp** function computed this way never overflows, and since one term of the sum is
exactly $e^0 = 1$ its logarithm is always finite. The cross-entropy of a label $y$ then becomes
$-\log \hat{p}_y = \mathrm{LSE}(\mathbf{z}) - z_y$, computed directly from the logits without ever
forming a probability. This is what `F.cross_entropy` and `F.log_softmax` do internally, and it is
why PyTorch's loss functions take logits rather than probabilities. A pleasant bonus: the gradient
of $\mathrm{LSE}$ is the softmax itself, $\partial \mathrm{LSE} / \partial z_i = e^{z_i} / \sum_k e^{z_k} = p_i$,
which the cell verifies with the gradient checker after demonstrating the failure of the naive
implementation.

In [ ]:
def log_sum_exp(z):
    """Stable log(sum(exp(z))) along the last axis."""
    c = z.max(axis=-1, keepdims=True)
    return (c + np.log(np.exp(z - c).sum(axis=-1, keepdims=True)))[..., 0]

def log_softmax(z):
    return z - log_sum_exp(z)[..., None]

def stable_softmax(z):
    return np.exp(log_softmax(z))

z_big = np.array([1000.0, 1000.0, 999.0])
with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
    naive = np.exp(z_big) / np.exp(z_big).sum()
    naive_log = np.log(naive)
print("naive softmax of [1000, 1000, 999]     :", naive, " <- overflow gives inf/inf = nan")
print("stable softmax                         :", stable_softmax(z_big))
print("stable log-softmax                     :", log_softmax(z_big))
z_wide = np.array([0.0, -800.0])
with np.errstate(divide="ignore"):
    print("naive log(softmax([0, -800]))          :", np.log(softmax(z_wide)), " <- underflow gives log(0) = -inf")
print("stable log-softmax                     :", log_softmax(z_wide))

z_rand = rng.standard_normal(5) * 3
gradient_check(log_sum_exp, stable_softmax, z_rand, name="gradient of LSE is the softmax")
print("shift invariance: softmax(z + 100) == softmax(z):", np.allclose(stable_softmax(z_rand + 100), stable_softmax(z_rand)))

Z_batch = rng.standard_normal((4, 3)) * 50           # a batch of 4 examples, 3 classes, large logits
labels = np.array([0, 2, 1, 2])
ce_per_example = log_sum_exp(Z_batch) - Z_batch[np.arange(4), labels]
print("batched cross-entropy from logits:", ce_per_example.round(3), "| mean:", ce_per_example.mean().round(3))
print("same via torch.nn.functional.cross_entropy:",
      F.cross_entropy(torch.tensor(Z_batch), torch.tensor(labels), reduction="none").cpu().numpy().round(3))

The naive implementation returns `nan` for large logits and $-\infty$ for widely separated ones; the
stable one returns the right answers in both cases, its gradient is the softmax to ten decimals, and
the batched cross-entropy computed from logits agrees with PyTorch's. Exercise 3 asks you to
implement the full softmax cross-entropy with its gradient in this stable form; it is the loss
function of the NumPy network in Notebook 02.

> **Key idea.** Losses are negative log-likelihoods: Gaussian noise gives MSE, Bernoulli or
> categorical outputs give cross-entropy, and minimising cross-entropy minimises the KL divergence
> between the data and the model. Always compute log-probabilities from logits with log-sum-exp.

## 5. Numerical computation

Everything so far assumed real numbers. Computers use floating-point numbers, and the difference
bites deep learning practitioners regularly: losses become `nan`, gradients vanish to exactly zero,
a variance comes out negative. This section gives the facts needed to predict and avoid such
problems, and closes with the two skills that make NumPy (and PyTorch) code fast: broadcasting and
vectorisation.

### 5.1 Floating-point formats: float64, float32, float16 and bfloat16

A floating-point number is stored as a sign, an exponent and a mantissa (fraction):
$x = (-1)^s \times 1.m \times 2^{e - \text{bias}}$. The number of exponent bits sets the **range**
(how large and how small a number can be) and the number of mantissa bits sets the **precision**
(how many significant digits). Two consequences follow. Numbers are spaced unevenly: the gap between
adjacent representable numbers grows with their magnitude, and the relative gap at 1, the **machine
epsilon** $\varepsilon$, is the precision of the format. And every arithmetic operation rounds its
result to the nearest representable number, introducing a relative error of up to $\varepsilon/2$.

| format | bits | exponent | mantissa | $\varepsilon$ | max | typical use |
|--------|-----:|---------:|---------:|--------------:|----:|-------------|
| float64 | 64 | 11 | 52 | $2.2 \times 10^{-16}$ | $1.8 \times 10^{308}$ | NumPy default; gradient checks |
| float32 | 32 | 8 | 23 | $1.2 \times 10^{-7}$ | $3.4 \times 10^{38}$ | PyTorch default for training |
| float16 | 16 | 5 | 10 | $9.8 \times 10^{-4}$ | $65\,504$ | mixed-precision training (needs loss scaling) |
| bfloat16 | 16 | 8 | 7 | $7.8 \times 10^{-3}$ | $3.4 \times 10^{38}$ | mixed precision on modern GPUs/TPUs |

The two 16-bit formats make different trades. `float16` keeps three decimal digits but overflows at
65 504, so a large activation or gradient becomes `inf`; `bfloat16` keeps the full range of
`float32` by sacrificing precision down to about two decimal digits. Notebook 14 · Efficient Training
and Scaling shows how mixed-precision training uses these formats, and why `float16` needs loss
scaling while `bfloat16` does not. The cell prints the numbers from NumPy's `finfo` (and PyTorch's,
since NumPy has no `bfloat16`), then demonstrates the classic consequences.

In [ ]:
print(f"{'format':<10}{'bits':>6}{'mantissa':>10}{'exponent':>10}{'epsilon':>12}{'max':>12}{'smallest normal':>18}")
for name, info in [("float16", np.finfo(np.float16)), ("float32", np.finfo(np.float32)), ("float64", np.finfo(np.float64))]:
    print(f"{name:<10}{info.bits:>6}{info.nmant:>10}{info.iexp:>10}{info.eps:>12.1e}{info.max:>12.1e}{info.smallest_normal:>18.1e}")
bf16 = torch.finfo(torch.bfloat16)
print(f"{'bfloat16':<10}{bf16.bits:>6}{7:>10}{8:>10}{bf16.eps:>12.1e}{bf16.max:>12.1e}{bf16.tiny:>18.1e}   (from torch.finfo)")

print("\n0.1 + 0.2 == 0.3 ?", 0.1 + 0.2 == 0.3, "  (0.1 + 0.2 =", repr(0.1 + 0.2), ")")
print("float32: 1e8 + 1 == 1e8 ?", np.float32(1e8) + np.float32(1) == np.float32(1e8), "  (spacing of float32 near 1e8 is", np.spacing(np.float32(1e8)), ")")
print("float32: 1 + 1e-8 == 1 ?", np.float32(1) + np.float32(1e-8) == np.float32(1), "  (below machine epsilon)")
with np.errstate(over="ignore"):
    print("float16: 60000 * 2 =", np.float16(60000) * np.float16(2), "  (overflow above 65504)")
print("float16: 1 + 1/1024 =", np.float16(1) + np.float16(1 / 1024), "| 1 + 1/2048 =", np.float16(1) + np.float16(1 / 2048), "  (10 mantissa bits)")
print("bfloat16: 1 + 1/128 =", torch.tensor(1 + 1 / 128, dtype=torch.bfloat16).item(),
      "| 1 + 1/256 =", torch.tensor(1 + 1 / 256, dtype=torch.bfloat16).item(), "  (7 mantissa bits)")
print("bfloat16: 60000 * 2 =", (torch.tensor(60000.0, dtype=torch.bfloat16) * 2).item(), "  (same range as float32)")

Each line is a rule of thumb you will need. Adding a small number to a large one loses the small
one entirely once their ratio drops below $\varepsilon$; in `float32` that happens at one part in
ten million, which is why learning rates below about $10^{-7}$ do nothing to weights of order one
and why very long sums (a loss over a huge batch, the running mean of a normalisation layer) should
be accumulated in `float32` or `float64`. `float16` overflows at a value that activations easily
reach; `bfloat16` does not, but cannot tell $1$ from $1 + 1/256$.

### 5.2 Overflow, underflow, catastrophic cancellation, and why we work with log-probabilities

Three failure modes account for most numerical bugs.

**Overflow** produces `inf` when a result exceeds the largest representable number, and `inf - inf`
or `inf / inf` then produces `nan`, which silently poisons every subsequent computation. The
exponential is the usual culprit, which is why softmax subtracts the maximum.

**Underflow** produces zero when a result is smaller than the smallest representable number. The
classic case is a product of probabilities: a likelihood of $N$ independent events is a product of
$N$ numbers below one and underflows long before $N$ reaches the size of a dataset. This, not merely
convenience, is why Section 4.4 worked with log-likelihoods: $\sum_i \log p_i$ is a perfectly
ordinary number when $\prod_i p_i$ is zero. Related: $\log \sigma(z)$ for a very negative logit
underflows inside the sigmoid before the logarithm sees it; the stable form is
$\log \sigma(z) = -\log(1 + e^{-z}) = -\mathrm{softplus}(-z)$, which is what `BCEWithLogitsLoss`
computes, and why PyTorch offers the "with logits" variants of the losses.

**Catastrophic cancellation** occurs when two nearly equal numbers are subtracted: the leading digits
cancel and only the rounding error in the trailing digits survives, so the *relative* error of the
result can be enormous even though each input was accurate. Textbook examples are $\log(1 + x)$ for
tiny $x$ (use `np.log1p`), $1 - \cos x$ for tiny $x$ (use $2\sin^2(x/2)$), and the "one-pass"
variance formula $\mathbb{E}[x^2] - \mathbb{E}[x]^2$, which subtracts two huge numbers when the data
have a large mean. The cell demonstrates each case, in `float32` where the effect is starkest.

In [ ]:
with np.errstate(over="ignore", under="ignore", divide="ignore", invalid="ignore"):
    print("overflow : exp(1000) =", np.exp(1000.0), "| exp(1000) - exp(1000) =", np.exp(1000.0) - np.exp(1000.0))
    print("underflow: exp(-1000) =", np.exp(-1000.0))
    p32 = np.full(200, 0.5, dtype=np.float32)
    print(f"product of 200 probabilities of 0.5 in float32: {np.prod(p32)} | sum of their logs: {np.sum(np.log(p32)):.3f} "
          f"(= 200 log 0.5 = {200 * np.log(0.5):.3f})")
    z_neg = np.float32(-100.0)
    print("log(sigmoid(-100)) naive:", np.log(1 / (1 + np.exp(-z_neg))), "| stable -softplus(-z) = -logaddexp(0, -z):", -np.logaddexp(np.float32(0), -z_neg))

tiny = 1e-10
print(f"\ncancellation: log(1 + 1e-10) naive = {np.log(1 + tiny):.12e} | log1p = {np.log1p(tiny):.12e} | exact series x - x^2/2 = {tiny - tiny ** 2 / 2:.12e}")
small = 1e-8
print(f"cancellation: 1 - cos(1e-8) naive = {1 - np.cos(small):.3e} | 2 sin^2(x/2) = {2 * np.sin(small / 2) ** 2:.3e} | exact x^2/2 = {small ** 2 / 2:.3e}")

data_shift = (1e4 + rng.standard_normal(10_000)).astype(np.float32)         # mean 10,000, true variance 1
one_pass = np.mean(data_shift ** 2) - np.mean(data_shift) ** 2                # E[x^2] - E[x]^2
two_pass = np.mean((data_shift - data_shift.mean()) ** 2)                      # E[(x - mean)^2]
print(f"variance of float32 data with mean 1e4: one-pass formula = {one_pass:.4f} | two-pass formula = {two_pass:.4f} | float64 reference = {np.var(data_shift.astype(np.float64)):.4f}")

The product of two hundred halves is exactly zero in `float32` while the sum of logs is a plain
$-138.6$; `log(sigmoid(-100))` is $-\infty$ naively and $-100$ stably; and the one-pass variance
formula, applied to data with mean $10^4$ in `float32`, returns a value with no correct digits (or
even a negative number, depending on the rounding), because $\mathbb{E}[x^2] \approx 10^8$ is
represented with an absolute error of order $10$ and the true variance is $1$. Whenever you compute
statistics of activations, do it the two-pass way, in `float32` or better; `BatchNorm` in
Notebook 04 · Training Deep Networks: Optimization, Initialization, Normalization, Regularization
does.

### 5.3 Broadcasting

**Broadcasting** is NumPy's rule for combining arrays of different shapes without copying data, and
PyTorch uses the same rule. To combine two arrays element-wise:

1. Align the shapes on the *right*, padding the shorter shape with ones on the left.
2. Two dimensions are compatible if they are equal, or if one of them is 1.
3. A dimension of size 1 is "stretched" (virtually repeated) to match the other; the result has the
   larger size in every dimension.
4. If some pair of dimensions is neither equal nor 1, the operation fails.

Shape diagrams make the rule mechanical. Adding a bias row to a batch:

```
X   (5, 3)
b      (3,)   ->  (1, 3)  ->  stretched to (5, 3)          OK: the bias is added to every row
```

Trying to add one number per row without reshaping:

```
X   (5, 3)
c      (5,)   ->  (1, 5)  ->  3 vs 5: incompatible          ERROR
c[:, None] (5, 1)          ->  stretched to (5, 3)          OK: one number per row
```

All pairwise differences between $N$ points and $M$ points in $d$ dimensions:

```
A[:, None, :]   (N, 1, d)
B[None, :, :]   (1, M, d)   ->  (N, M, d)                  every point of A minus every point of B
```

The last pattern, inserting a length-1 axis with `None` (or `np.newaxis`) so that two arrays
broadcast against each other, is the workhorse of vectorised code: it replaces a double loop with a
single expression. The cell works through these cases, including the error, and shows the two most
common applications in this course: standardising the columns of a data matrix, and computing a
matrix of pairwise squared distances.

In [ ]:
X5 = np.arange(15, dtype=np.float64).reshape(5, 3)
b3 = np.array([10.0, 20.0, 30.0])
c5 = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
print("X + b     : shapes", X5.shape, "+", b3.shape, "->", (X5 + b3).shape, "  (b added to every row)")
try:
    X5 + c5
except ValueError as err:
    print("X + c     : ValueError ->", err)
print("X + c[:,None]: shapes", X5.shape, "+", c5[:, None].shape, "->", (X5 + c5[:, None]).shape, "  (one number per row)")
print("outer product via broadcasting: c5[:, None] * b3[None, :] has shape", (c5[:, None] * b3[None, :]).shape,
      "| equals np.outer:", np.allclose(c5[:, None] * b3[None, :], np.outer(c5, b3)))

# standardising features: subtract each column's mean, divide by its standard deviation
X_data = rng.normal(loc=[100.0, 0.0, -5.0], scale=[10.0, 1.0, 0.1], size=(1000, 3))
X_std = (X_data - X_data.mean(axis=0)) / X_data.std(axis=0)       # (1000, 3) - (3,) / (3,)
print("\nstandardised column means:", X_std.mean(axis=0).round(6), "| stds:", X_std.std(axis=0).round(6))

# pairwise squared distances between N points A and M points B
A_pts, B_pts = rng.standard_normal((4, 2)), rng.standard_normal((3, 2))
diff = A_pts[:, None, :] - B_pts[None, :, :]                      # (4, 1, 2) - (1, 3, 2) -> (4, 3, 2)
D2 = (diff ** 2).sum(axis=-1)                                      # (4, 3)
D2_loop = np.array([[np.sum((a - b) ** 2) for b in B_pts] for a in A_pts])
print("pairwise squared distances: shape", D2.shape, "| equals the double loop:", np.allclose(D2, D2_loop))
print("keepdims: X_data.mean(axis=1).shape =", X_data.mean(axis=1).shape, "| with keepdims=True:", X_data.mean(axis=1, keepdims=True).shape,
      " <- keepdims keeps the axis so the result broadcasts back against X_data")

The error message for the failing case ("operands could not be broadcast together with shapes
(5,3) (5,)") is one you will see often; the fix is almost always a `[:, None]` or a
`keepdims=True`. The standardisation and pairwise-distance idioms reappear in PyTorch syntax
(`x.unsqueeze(1)` plays the role of `x[:, None]`) in Notebook 02 · Neural Networks from Scratch
(NumPy), Notebook 05 · The PyTorch Workflow and Best Practices and Notebook 11 · Autoencoders and
Variational Autoencoders.

### 5.4 Vectorisation versus Python loops

Python is slow at arithmetic: each `+` in a loop involves type checks, object allocation and
interpreter overhead, roughly a hundred nanoseconds to a microsecond per operation. NumPy's array
operations run the same arithmetic in compiled C over contiguous memory, using the processor's
vector instructions and, for matrix products, multi-threaded BLAS libraries. The difference is
routinely a factor of a hundred, and on a GPU with PyTorch it is thousands. **Vectorisation** means
writing the computation as whole-array operations with no Python loop over the data. Section 1.4's
batched layer and Section 5.3's pairwise distances are already vectorised; the cell measures what
that buys, on a dot product and on the pairwise-distance matrix, and adds a third variant of the
latter based on the identity $\lVert \mathbf{a} - \mathbf{b} \rVert^2 = \lVert \mathbf{a} \rVert^2 +
\lVert \mathbf{b} \rVert^2 - 2\mathbf{a}^\top\mathbf{b}$, which turns the whole thing into one matrix
product (and is how nearest-neighbour searches and attention-like scores are computed at scale).

In [ ]:
n_dot = 200_000 if FAST_RUN else 1_000_000
u_vec, v_vec = rng.standard_normal(n_dot), rng.standard_normal(n_dot)

def dot_with_loop(u, v):
    acc = 0.0
    for i in range(len(u)):
        acc += u[i] * v[i]
    return acc

t_loop = best_of(lambda: dot_with_loop(u_vec, v_vec), repeats=2)
t_ufunc = best_of(lambda: np.sum(u_vec * v_vec), repeats=10)        # element-wise product + sum (single-threaded ufuncs)
t_blas = best_of(lambda: u_vec @ v_vec, repeats=10)                  # BLAS dot product
print(f"dot product of {n_dot:,} numbers: Python loop {t_loop * 1e3:8.1f} ms | np.sum(u * v) {t_ufunc * 1e3:7.3f} ms "
      f"({t_loop / t_ufunc:,.0f}x faster) | u @ v {t_blas * 1e3:7.3f} ms ({t_loop / t_blas:,.0f}x faster) | "
      f"same result: {np.isclose(dot_with_loop(u_vec, v_vec), u_vec @ v_vec)}")

n_pts = 200 if FAST_RUN else 500
P = rng.standard_normal((n_pts, 10))

def pairwise_loop(P):
    D = np.empty((len(P), len(P)))
    for i in range(len(P)):
        for j in range(len(P)):
            D[i, j] = np.sum((P[i] - P[j]) ** 2)
    return D

def pairwise_broadcast(P):
    return ((P[:, None, :] - P[None, :, :]) ** 2).sum(axis=-1)

def pairwise_matmul(P):
    sq = (P ** 2).sum(axis=1)
    return np.maximum(sq[:, None] + sq[None, :] - 2 * P @ P.T, 0.0)       # clip tiny negatives from rounding

results = {}
for name, fn in [("double loop", pairwise_loop), ("broadcasting", pairwise_broadcast), ("matmul identity", pairwise_matmul)]:
    t0 = time.perf_counter(); D = fn(P); elapsed = time.perf_counter() - t0
    results[name] = (D, elapsed)
    print(f"pairwise distances of {n_pts} points, {name:<16}: {elapsed * 1e3:8.2f} ms")
print("all three agree:", all(np.allclose(results["double loop"][0], D, atol=1e-8) for D, _ in results.values()))

The loop is slower than the array expressions by two to three orders of magnitude on an idle
machine (the BLAS call `u @ v` is usually the fastest of all, although on a heavily loaded machine
its worker threads can make it slower than the single-threaded `np.sum(u * v)` for inputs this
small), and the broadcasting version of the distance matrix is faster than the double loop by one to two
orders of magnitude; the matrix-product identity is faster still because it uses BLAS and touches far
less memory than the $(N, N, d)$ intermediate of the broadcasting version. The rule for the rest of the course: **if you
find yourself writing `for i in range(len(X))` over data, stop and look for the array expression.**
The one exception is the loop over mini-batches, whose body is itself vectorised.

> **Key idea.** Floating-point numbers have finite range and precision; use log-probabilities, stable
> formulas and two-pass statistics; express computations as broadcasted array operations rather than
> loops.

## Exercises

The exercises use the helpers defined above: `numerical_gradient`, `gradient_check`,
`gradient_descent`, `log_sum_exp`, `log_softmax` and `stable_softmax`. Write your own attempt in a
new cell before opening the solution.

### Exercise 1 (warm-up): PCA on the iris dataset

Load the iris dataset (`load_iris`, 150 flowers described by four measurements in centimetres,
three species). Centre the data, compute its SVD, and (a) plot the scores on the first two principal
components coloured by species; (b) print the variance explained by each component; (c) reconstruct
the data from $k = 1, 2, 3, 4$ components and report the relative reconstruction error
$\lVert \mathbf{X}_c - \hat{\mathbf{X}}_k \rVert_F / \lVert \mathbf{X}_c \rVert_F$, verifying that it
equals $\sqrt{1 - (\text{explained variance of the first } k)}$.

**Solution to Exercise 1**

The recipe is the one of Section 1.9: centre, `np.linalg.svd`, scores $\mathbf{Z} = \mathbf{X}_c \mathbf{V}$,
reconstruction $\hat{\mathbf{X}}_k = \mathbf{Z}_{:,:k} \mathbf{V}_{:,:k}^\top$. Because the four
measurements are in the same unit we work with the centred but unscaled data; when features have
different units one usually standardises each column first (Section 5.3), which changes the
components. The first component alone carries over 92% of the variance, dominated by petal length
(the measurement with by far the largest spread), and separates one species completely from the
other two. The reconstruction error follows
the Eckart–Young prediction exactly, as it must: PCA *is* the truncated SVD of the centred data.

In [ ]:
X_iris, y_iris = load_iris(return_X_y=True)
Xc_iris = X_iris - X_iris.mean(axis=0)
U_i, s_i, Vt_i = np.linalg.svd(Xc_iris, full_matrices=False)
scores_iris = Xc_iris @ Vt_i.T
explained_iris = s_i ** 2 / (s_i ** 2).sum()
print("explained variance per component:", explained_iris.round(4), "| cumulative:", np.cumsum(explained_iris).round(4))
print("first principal direction (sepal length, sepal width, petal length, petal width):", Vt_i[0].round(3))

for k in range(1, 5):
    X_hat = (Xc_iris @ Vt_i[:k].T) @ Vt_i[:k]
    rel_err = np.linalg.norm(Xc_iris - X_hat, "fro") / np.linalg.norm(Xc_iris, "fro")
    predicted = np.sqrt(max(1 - explained_iris[:k].sum(), 0.0))
    print(f"k = {k}: relative reconstruction error {rel_err:.4f} | sqrt(1 - explained) = {predicted:.4f}")

fig, ax = plt.subplots(figsize=(6, 4.5))
for species, marker in zip(range(3), ["o", "s", "^"]):
    mask = y_iris == species
    ax.scatter(scores_iris[mask, 0], scores_iris[mask, 1], marker=marker, s=25, label=f"species {species}")
ax.set_xlabel(f"PC 1 ({explained_iris[0]:.1%} of variance)"); ax.set_ylabel(f"PC 2 ({explained_iris[1]:.1%})")
ax.set_title("Iris on its first two principal components"); ax.legend(); plt.tight_layout(); plt.show()

### Exercise 2: gradient-check a hand-derived gradient

Logistic regression with labels $y_i \in \{-1, +1\}$ and an $\ell_2$ penalty minimises

$$
L(\mathbf{w}) = \frac{1}{N} \sum_{i=1}^{N} \log\big(1 + e^{-y_i \mathbf{x}_i^\top \mathbf{w}}\big) + \frac{\lambda}{2} \lVert \mathbf{w} \rVert_2^2 .
$$

(a) Derive $\nabla_{\mathbf{w}} L$ by hand. (b) Implement the loss in a numerically stable way (hint:
`np.logaddexp(0, -m)` computes $\log(1 + e^{-m})$ without overflow) and the gradient, and validate
it with `gradient_check` on random data. (c) Introduce a deliberate sign error and confirm that the
checker catches it. (d) Run gradient descent on a small synthetic dataset and report the training
accuracy.

**Solution to Exercise 2**

Let $m_i = y_i \mathbf{x}_i^\top \mathbf{w}$ be the *margin*. The chain rule gives
$\frac{d}{dm} \log(1 + e^{-m}) = \frac{-e^{-m}}{1 + e^{-m}} = -\sigma(-m)$, and
$\nabla_{\mathbf{w}} m_i = y_i \mathbf{x}_i$, so

$$
\nabla_{\mathbf{w}} L = -\frac{1}{N} \sum_{i=1}^{N} \sigma(-m_i)\, y_i \mathbf{x}_i + \lambda \mathbf{w}
= -\frac{1}{N} \mathbf{X}^\top \big( \boldsymbol{\sigma}(-\mathbf{m}) \odot \mathbf{y} \big) + \lambda \mathbf{w}.
$$

The factor $\sigma(-m_i)$ is the probability the model assigns to the *wrong* label, so each example
pulls on the weights in proportion to how badly it is classified, a pattern that recurs in every
classification gradient in the course. The stable implementation uses `np.logaddexp` for the loss
and `1 / (1 + exp(m))` for $\sigma(-m)$, which is safe because a large positive $m$ merely gives
zero. The checker returns about $10^{-10}$ for the correct gradient and about $1$ for the version
with the sign flipped.

In [ ]:
def logistic_loss(w, X, y, lam):
    margins = y * (X @ w)
    return np.mean(np.logaddexp(0.0, -margins)) + 0.5 * lam * w @ w

def logistic_grad(w, X, y, lam):
    margins = y * (X @ w)
    prob_wrong = 1.0 / (1.0 + np.exp(margins))            # sigma(-m)
    return -(X.T @ (prob_wrong * y)) / len(y) + lam * w

def logistic_grad_buggy(w, X, y, lam):
    margins = y * (X @ w)
    prob_wrong = 1.0 / (1.0 + np.exp(margins))
    return (X.T @ (prob_wrong * y)) / len(y) + lam * w     # sign error on the data term

n_lr, d_lr, lam = 200, 5, 1e-2
w_true = rng.standard_normal(d_lr)
X_lr = rng.standard_normal((n_lr, d_lr))
y_lr = np.where(X_lr @ w_true + 0.5 * rng.standard_normal(n_lr) > 0, 1.0, -1.0)
w_check = rng.standard_normal(d_lr)
gradient_check(lambda w: logistic_loss(w, X_lr, y_lr, lam), lambda w: logistic_grad(w, X_lr, y_lr, lam), w_check, name="logistic regression, correct")
gradient_check(lambda w: logistic_loss(w, X_lr, y_lr, lam), lambda w: logistic_grad_buggy(w, X_lr, y_lr, lam), w_check, name="logistic regression, sign bug")

path = gradient_descent(lambda w: logistic_grad(w, X_lr, y_lr, lam), np.zeros(d_lr), lr=0.5, n_steps=300)
losses = [logistic_loss(w, X_lr, y_lr, lam) for w in path]
w_fit = path[-1]
train_acc = np.mean(np.sign(X_lr @ w_fit) == y_lr)
print(f"loss: start {losses[0]:.4f} (= log 2 = {np.log(2):.4f}) -> end {losses[-1]:.4f} | training accuracy {train_acc:.3f}")
print(f"cosine similarity between the fitted and the true weight vector: {w_fit @ w_true / np.linalg.norm(w_fit) / np.linalg.norm(w_true):.4f}")

The initial loss is $\log 2$, the loss of predicting probability one half for everything, another
sanity-check value worth remembering. After three hundred steps the fitted weights point almost
exactly along the true ones, and the residual errors come from the label noise we added, not from
the optimizer.

### Exercise 3: a stable softmax cross-entropy with its gradient

Implement `softmax_cross_entropy(logits, labels)` for a batch of logits of shape `(N, K)` and integer
labels of shape `(N,)`, returning the *mean* loss and its gradient with respect to the logits, using
`log_sum_exp` so that it works for arbitrarily large logits. Derive the gradient
$\partial L / \partial \mathbf{z}_i = (\hat{\mathbf{p}}_i - \mathbf{y}_i) / N$, with $\mathbf{y}_i$ the
one-hot label. Validate it against `F.cross_entropy` and against `gradient_check`, including a batch
with logits of magnitude $10^4$.

**Solution to Exercise 3**

The per-example loss is $\ell_i = \mathrm{LSE}(\mathbf{z}_i) - z_{i, y_i}$. Differentiating with
respect to $z_{ij}$: the LSE term gives $\partial \mathrm{LSE} / \partial z_{ij} = \hat{p}_{ij}$
(Section 4.6), and the second term gives $-1$ if $j = y_i$ and $0$ otherwise, so
$\partial \ell_i / \partial \mathbf{z}_i = \hat{\mathbf{p}}_i - \mathbf{y}_i$. Averaging over the batch
divides by $N$. The implementation never forms the softmax except through `exp(log_softmax)`, which
is safe, and subtracts one from the correct class's probability by indexing rather than by building
a one-hot matrix. The gradient check treats the whole `(N, K)` logit matrix as the variable; the
values match PyTorch's `F.cross_entropy` in both the moderate and the extreme case.

In [ ]:
def softmax_cross_entropy(logits, labels):
    """Mean cross-entropy of integer labels given logits (N, K), and its gradient w.r.t. the logits."""
    N = logits.shape[0]
    log_probs = log_softmax(logits)                                   # (N, K), stable
    loss = -np.mean(log_probs[np.arange(N), labels])
    grad = np.exp(log_probs)                                          # softmax probabilities
    grad[np.arange(N), labels] -= 1.0
    return loss, grad / N

for scale, description in [(1.0, "moderate logits"), (1e4, "logits of magnitude 1e4")]:
    Z = rng.standard_normal((6, 4)) * scale
    labels_z = rng.integers(0, 4, size=6)
    loss_np, grad_np = softmax_cross_entropy(Z, labels_z)
    Z_t = torch.tensor(Z, requires_grad=True)
    loss_t = F.cross_entropy(Z_t, torch.tensor(labels_z))
    loss_t.backward()
    print(f"{description:<26} loss numpy {loss_np:.6f} | torch {loss_t.item():.6f} | max |grad difference| {np.abs(grad_np - Z_t.grad.cpu().numpy()).max():.2e}")
    gradient_check(lambda Zz: softmax_cross_entropy(Zz, labels_z)[0], lambda Zz: softmax_cross_entropy(Zz, labels_z)[1], Z,
                   h=1e-5 * max(1.0, scale / 100), name=f"  gradient check ({description})")

Note the larger finite-difference step for the extreme case. The rounding error of a central
difference is about $\varepsilon |f| / h$ (Section 2.2), and here $|f| \approx 10^4$, so with
$h = 10^{-5}$ it would be around $10^{-7}$, at the edge of the "correct" band of the checker even
though the gradient is right. Scaling $h$ with the magnitude of the variable keeps the check
meaningful, a detail that gradient checks on real networks (where weights and activations have
very different scales) also require. This function is the loss of the NumPy MLP in Notebook 02 ·
Neural Networks from Scratch (NumPy).

### Exercise 4: a vectorised nearest-neighbour classifier

Split the digits data into 1500 training and 297 test images. Implement a $k$-nearest-neighbour
classifier ($k = 5$, majority vote) whose distance computation uses the matrix-product identity of
Section 5.4 and whose voting uses broadcasting, with no Python loop over examples. Report the test
accuracy, and compare the running time with a loop-based implementation on a subset of the test set.

**Solution to Exercise 4**

The squared distances between all test and all training points are one matrix product plus two
broadcasted norms. `np.argsort` along the last axis gives the indices of the nearest neighbours,
`labels_train[neighbours]` gathers their labels into an `(n_test, k)` array, and a one-hot encoding
`np.eye(10)[...]` summed over the neighbour axis counts the votes for every class in a single
operation. The result is about 98–99% test accuracy in a few tens of milliseconds, versus hundreds
of milliseconds to seconds for the loop on a fraction of the queries, and the two agree on their
predictions. The vectorised version also has an obvious weakness: the distance matrix grows with
(test size) × (training size), so for very large datasets one processes the test set in chunks, the
same batching logic as everywhere else in the course.

In [ ]:
X_dig, y_dig = load_digits(return_X_y=True)
perm = rng.permutation(len(X_dig))
train_idx, test_idx = perm[:1500], perm[1500:]
X_tr, y_tr, X_te, y_te = X_dig[train_idx], y_dig[train_idx], X_dig[test_idx], y_dig[test_idx]

def knn_predict_vectorised(X_train, y_train, X_query, k=5, n_classes=10):
    sq_train, sq_query = (X_train ** 2).sum(axis=1), (X_query ** 2).sum(axis=1)
    D2 = sq_query[:, None] + sq_train[None, :] - 2 * X_query @ X_train.T         # (n_query, n_train)
    neighbours = np.argsort(D2, axis=1)[:, :k]                                     # (n_query, k)
    votes = np.eye(n_classes)[y_train[neighbours]].sum(axis=1)                     # (n_query, k, C) -> (n_query, C)
    return votes.argmax(axis=1)

def knn_predict_loop(X_train, y_train, X_query, k=5):
    predictions = []
    for q in X_query:
        dists = [np.sum((q - x) ** 2) for x in X_train]
        nearest = np.argsort(dists)[:k]
        predictions.append(np.bincount(y_train[nearest], minlength=10).argmax())
    return np.array(predictions)

t0 = time.perf_counter(); pred_vec = knn_predict_vectorised(X_tr, y_tr, X_te); t_vec = time.perf_counter() - t0
print(f"vectorised 5-NN: test accuracy {np.mean(pred_vec == y_te):.4f} on {len(y_te)} images in {t_vec * 1e3:.1f} ms")

n_sub = 60 if FAST_RUN else 150
t0 = time.perf_counter(); pred_loop = knn_predict_loop(X_tr, y_tr, X_te[:n_sub]); t_loop = time.perf_counter() - t0
t0 = time.perf_counter(); pred_vec_sub = knn_predict_vectorised(X_tr, y_tr, X_te[:n_sub]); t_vec_sub = time.perf_counter() - t0
print(f"loop version on {n_sub} queries: {t_loop * 1e3:.0f} ms | vectorised: {t_vec_sub * 1e3:.1f} ms | speed-up {t_loop / t_vec_sub:.0f}x | "
      f"identical predictions: {np.array_equal(pred_loop, pred_vec_sub)}")

### Exercise 5 (challenge): fitting a Gaussian mixture by gradient descent on the likelihood

Sample 900 points in the plane from a mixture of three Gaussians with equal weights, known standard
deviation $\sigma = 0.7$ and means $(0, 0)$, $(4, 1)$, $(1, 4)$. Treat the means as unknown and
recover them by gradient descent on the negative log-likelihood

$$
\mathrm{NLL}(\boldsymbol{\mu}_{1:3}) = -\sum_{n=1}^{N} \log \sum_{k=1}^{3} \pi_k\, \mathcal{N}(\mathbf{x}_n \mid \boldsymbol{\mu}_k, \sigma^2 \mathbf{I}) .
$$

Derive the gradient, implement the NLL with `log_sum_exp` (the sum inside the logarithm is a
log-sum-exp over the components), validate the gradient with `gradient_check`, run gradient descent
from a random initialisation, and plot the trajectories of the three means on top of the data.

**Solution to Exercise 5**

Write $a_{nk} = \log \pi_k + \log \mathcal{N}(\mathbf{x}_n \mid \boldsymbol{\mu}_k, \sigma^2\mathbf{I})$,
with $\log \mathcal{N} = -\lVert \mathbf{x}_n - \boldsymbol{\mu}_k \rVert^2 / (2\sigma^2) - \log(2\pi\sigma^2)$
in two dimensions. Then $\mathrm{NLL} = -\sum_n \mathrm{LSE}_k(a_{nk})$, and by the LSE-gradient rule
of Section 4.6 the derivative of $\mathrm{LSE}_k(a_{nk})$ with respect to $a_{nk}$ is the softmax
$r_{nk} = \mathrm{softmax}_k(a_{n\cdot})$, the **responsibility** of component $k$ for point $n$.
Since $\partial a_{nk} / \partial \boldsymbol{\mu}_k = (\mathbf{x}_n - \boldsymbol{\mu}_k) / \sigma^2$,

$$
\frac{\partial\, \mathrm{NLL}}{\partial \boldsymbol{\mu}_k} = -\frac{1}{\sigma^2} \sum_{n=1}^{N} r_{nk}\, (\mathbf{x}_n - \boldsymbol{\mu}_k) .
$$

Setting this to zero gives $\boldsymbol{\mu}_k = \sum_n r_{nk} \mathbf{x}_n / \sum_n r_{nk}$, the
responsibility-weighted mean, which is exactly the M-step of the EM algorithm; gradient descent on
the NLL takes small steps towards that fixed point, with the responsibilities re-computed at every
step. We optimise the mean NLL per point (so that the learning rate does not depend on $N$) and
flatten the three means into a single vector so that `gradient_descent` and `gradient_check` can be
reused unchanged. The mixture likelihood is non-convex (Section 3.4): a poor initialisation can leave
two means sharing one cluster, so we initialise at three random data points, as k-means does.

In [ ]:
sigma_mix = 0.7
true_means = np.array([[0.0, 0.0], [4.0, 1.0], [1.0, 4.0]])
n_per = 300
X_mix = np.concatenate([rng.normal(m, sigma_mix, size=(n_per, 2)) for m in true_means])
log_pi = np.log(np.full(3, 1 / 3))

def mixture_log_joint(means_flat, X):
    """a[n, k] = log pi_k + log N(x_n | mu_k, sigma^2 I), shape (N, 3)."""
    means = means_flat.reshape(3, 2)
    sq = ((X[:, None, :] - means[None, :, :]) ** 2).sum(axis=-1)          # (N, 3)
    return log_pi[None, :] - sq / (2 * sigma_mix ** 2) - np.log(2 * np.pi * sigma_mix ** 2)

def mixture_mean_nll(means_flat, X=X_mix):
    return -np.mean(log_sum_exp(mixture_log_joint(means_flat, X)))

def mixture_mean_nll_grad(means_flat, X=X_mix):
    means = means_flat.reshape(3, 2)
    resp = np.exp(log_softmax(mixture_log_joint(means_flat, X)))            # responsibilities r[n, k]
    grad = -(resp[:, :, None] * (X[:, None, :] - means[None, :, :])).sum(axis=0) / sigma_mix ** 2 / len(X)
    return grad.ravel()

gradient_check(lambda m: mixture_mean_nll(m, X_mix[:100]), lambda m: mixture_mean_nll_grad(m, X_mix[:100]),
               rng.standard_normal(6), name="mixture NLL gradient")

init = X_mix[rng.choice(len(X_mix), size=3, replace=False)].ravel()          # three random data points
path = gradient_descent(mixture_mean_nll_grad, init, lr=1.0, n_steps=150)
nll_curve = [mixture_mean_nll(m) for m in path]
fitted = path[-1].reshape(3, 2)
print("fitted means:\n", fitted.round(3))
print("mean NLL: start", round(nll_curve[0], 4), "-> end", round(nll_curve[-1], 4), "| at the true means:", round(mixture_mean_nll(true_means.ravel()), 4))
matching = [np.linalg.norm(true_means - f, axis=1).min() for f in fitted]
print("distance from each fitted mean to the nearest true mean:", np.round(matching, 3))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.6))
ax1.scatter(X_mix[:, 0], X_mix[:, 1], s=6, alpha=0.4, color="gray", label="data")
for k, color in enumerate(["C0", "C1", "C3"]):
    traj = path[:, 2 * k: 2 * k + 2]
    ax1.plot(traj[:, 0], traj[:, 1], "-", color=color, lw=2)
    ax1.scatter(traj[0, 0], traj[0, 1], marker="o", color=color, s=60, edgecolors="k", zorder=3)
    ax1.scatter(traj[-1, 0], traj[-1, 1], marker="s", color=color, s=60, edgecolors="k", zorder=3)
ax1.scatter(true_means[:, 0], true_means[:, 1], marker="*", s=250, color="gold", edgecolors="k", zorder=4, label="true means")
ax1.set_aspect("equal"); ax1.set_title("Trajectories of the three means (circle = start, square = end)"); ax1.legend(fontsize=8)
ax2.plot(nll_curve); ax2.axhline(mixture_mean_nll(true_means.ravel()), color="k", ls="--", label="NLL at the true means")
ax2.set_xlabel("gradient step"); ax2.set_ylabel("mean negative log-likelihood"); ax2.set_title("The likelihood improves monotonically"); ax2.legend()
plt.tight_layout(); plt.show()

The three means travel from their random starting points to the centres of the clusters within a
few dozen steps and the final NLL matches the NLL at the true means (it can be marginally lower,
since the sample means of finite clusters differ slightly from the population means). The gradient
we derived is, up to the learning rate, the EM update; Notebook 11 · Autoencoders and Variational
Autoencoders generalises this idea to latent variables that cannot be summed out, which is where the
evidence lower bound comes from.

## Summary

- Vectors, matrices and tensors are NumPy arrays distinguished by shape; a matrix is a linear map,
  its columns are the images of the basis vectors, and $(m \times n)(n \times p) = (m \times p)$ is the
  shape rule behind every layer. Batching turns $N$ matrix–vector products into one matrix–matrix
  product.
- The dot product measures alignment ($\mathbf{a}^\top\mathbf{b} = \lVert\mathbf{a}\rVert\lVert\mathbf{b}\rVert\cos\theta$)
  and projects one vector onto another; a neuron is a projection onto its weight vector.
- Norms measure size and drive regularisation ($\ell_2$ shrinks, $\ell_1$ sparsifies). Symmetric
  matrices have real eigenvalues and orthogonal eigenvectors; the SVD generalises this to any matrix,
  gives the best low-rank approximation (Eckart–Young), and applied to centred data is PCA.
- The derivative is the best linear approximation; the gradient points in the direction of steepest
  ascent and is perpendicular to the contours; the Jacobian is the derivative of a vector function and
  the chain rule multiplies Jacobians. For a scalar loss, evaluating the chain right-to-left as
  vector–Jacobian products is backpropagation, with $\partial L/\partial\mathbf{W} = \mathbf{X}^\top\mathbf{G}$
  and $\partial L/\partial\mathbf{X} = \mathbf{G}\mathbf{W}^\top$ for a linear layer.
- Central differences have error $O(h^2)$ and should be used, in `float64` with $h \approx 10^{-5}$,
  to check every hand-derived gradient via the relative error.
- Gradient descent decreases the loss because of the first-order Taylor term; the learning rate must
  be below $2/\lambda_{\max}$ of the Hessian, and the condition number $\lambda_{\max}/\lambda_{\min}$
  sets the number of steps. Non-convex losses have local minima, saddle points and plateaus;
  momentum reduces the cost of ill-conditioning from $O(\kappa)$ to $O(\sqrt{\kappa})$.
- Losses are negative log-likelihoods: Gaussian noise yields mean-squared error, Bernoulli and
  categorical outputs yield cross-entropy, and minimising cross-entropy minimises the KL divergence
  between the data and the model.
- Entropy measures uncertainty, cross-entropy the expected surprise under a wrong model, and their
  difference is the (asymmetric, non-negative) KL divergence.
- Softmax is shift-invariant; compute log-probabilities as $z_i - \mathrm{LSE}(\mathbf{z})$ with the
  max subtracted, and cross-entropy directly from logits.
- Floating point has finite range and precision: watch for overflow, underflow and cancellation;
  use log-probabilities, `log1p`, two-pass statistics, `float32` accumulators. Broadcasting aligns
  shapes from the right; vectorised array code is orders of magnitude faster than Python loops.

## Further reading

- Goodfellow, I., Bengio, Y. and Courville, A. (2016). *Deep Learning*, MIT Press. Chapters 2
  (linear algebra), 3 (probability and information theory) and 4 (numerical computation) cover the
  same ground as this notebook in more depth.
- Deisenroth, M. P., Faisal, A. A. and Ong, C. S. (2020). *Mathematics for Machine Learning*.
  Cambridge University Press (free online). The best single book for the prerequisites, with
  chapters on vector calculus and PCA.
- Strang, G. (2019). *Linear Algebra and Learning from Data*. Wellesley-Cambridge Press. The SVD and
  low-rank viewpoint of Section 1, by its most eloquent advocate.
- Bishop, C. M. (2006). *Pattern Recognition and Machine Learning*. Springer. Chapters 1 and 2 for
  probability, maximum likelihood and the exponential family; Chapter 9 for mixtures and EM.
- Murphy, K. P. (2022). *Probabilistic Machine Learning: An Introduction*. MIT Press (free online).
  A modern, comprehensive treatment of the probabilistic view of Section 4.
- Nocedal, J. and Wright, S. J. (2006). *Numerical Optimization*, 2nd ed. Springer. Chapters 2 and 3
  for the theory behind Section 3 (line search, convergence of gradient descent, conditioning).
- Polyak, B. T. (1964). Some methods of speeding up the convergence of iteration methods. *USSR
  Computational Mathematics and Mathematical Physics*, 4(5), 1–17. The heavy-ball method of Section 3.5.
- Dauphin, Y. N. et al. (2014). Identifying and attacking the saddle point problem in high-dimensional
  non-convex optimization. *NeurIPS*. Why saddle points, not local minima, dominate.
- Goldberg, D. (1991). What every computer scientist should know about floating-point arithmetic.
  *ACM Computing Surveys*, 23(1), 5–48. The classic reference for Section 5.
- Higham, N. J. (2002). *Accuracy and Stability of Numerical Algorithms*, 2nd ed. SIAM. Cancellation,
  summation and the analysis of rounding errors.
- NumPy documentation on broadcasting: https://numpy.org/doc/stable/user/basics.broadcasting.html,
  and the PyTorch documentation, https://docs.pytorch.org/docs/stable/, whose broadcasting semantics
  page follows the same rules.